# Chapter 05 제출 답안. 분석을 믿을 수 있게 만드는 데이터 전처리

## 0. 제출 정보
- 이름: 장태진
- GitHub ID: Taejin-AI
- 작성일: 2026.10.15
- 최종 제출 URL: https://github.com/Taejin-AI/llm-data-analysis-study/blob/master/chapter05/chapter05.ipynb
- 함께 제출한 파일: `chapter05/ch05_preprocess.py` (이 Notebook의 전처리 규칙을 명령 하나로 재현하는 스크립트, 6번에서 검증), `chapter05/images/` (Evidence 7장)

## 1. 전처리 전 상태 기록

### 전처리 대상과 범위
- 사용 데이터: `practice/chapter05/data/raw/`의 `customers.csv`, `products.csv`, `orders.csv`, `order_items.csv` 4개입니다.
  - 공식 스크립트 `scripts/preprocess_data.py`가 읽는 입력과 같은 파일입니다.
  - 공통 `data/raw/`는 Chapter 04에서 결측·중복·0 이하 값이 모두 0건으로 확인되어 전처리 판단을 연습할 수 없으므로 쓰지 않았습니다.
- 범위: 4개 파일의 **모든 행(전수)**을 전처리합니다. 전처리 단계에서는 기간으로 자르지 않습니다. 날짜 검증 결과(2번)에 따라 이후 분석에 쓸 수 있는 주문일 범위는 **2025-07-01 ~ 2026-08-30**입니다(변환 실패 3건과 작성일 이후 미래 날짜 2건은 플래그 처리).
- 원칙: 원본 파일은 읽기만 합니다. 원본 파일의 MD5 해시를 처음과 마지막에 비교해 바뀌지 않았는지 확인합니다(6번).

### 테이블 연결 관계와 전처리 영향
| 테이블 | PK | FK (→ 부모) | 이 테이블의 전처리가 영향을 주는 곳 |
| --- | --- | --- | --- |
| `customers` | `customer_id` | - | 행을 지우면 그 고객의 `orders` 행이 참조 대상이 없는 행(orphan)이 됨 |
| `products` | `product_id` | - | 행을 지우면 `order_items` 행이 orphan이 됨 → 카테고리 집계에서 거래 누락 |
| `orders` | `order_id` | `customer_id` → `customers` | 행을 지우면 `order_items` 행이 orphan이 됨. 상태·주문일 필터의 기준 |
| `order_items` | `order_item_id` | `order_id` → `orders`, `product_id` → `products` | 상품 주문 금액(`quantity × unit_price`)의 기준 행 |

- 부모 테이블(`customers`, `products`, `orders`)에서 행을 지우면 자식 테이블의 관계가 끊길 수 있습니다. 그래서 PK/FK를 전처리 전(이 절)과 후(4번)에 모두 확인합니다.
- `customers.name`은 어떤 전처리 규칙에도 필요 없는 개인정보라서 출력·저장하지 않습니다.

In [1]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys

import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'data' / 'raw'   # Chapter 05 전용 Raw (scripts/preprocess_data.py 입력과 같음)
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'                   # 공식 스크립트 결과 위치
OUT_DIR = PROCESSED_DIR / 'ch05_notebook'                             # 내 규칙(이 Notebook) 결과 위치
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)
print('프로젝트 루트:', PROJECT_ROOT.name)
print('원본 데이터 폴더:', RAW_DIR.relative_to(PROJECT_ROOT), '| 존재:', RAW_DIR.exists())
print('결과 폴더: 공식 스크립트 →', PROCESSED_DIR.relative_to(PROJECT_ROOT), '| 내 규칙 →', OUT_DIR.relative_to(PROJECT_ROOT))

Python: 3.13.16 | pandas: 3.0.5
프로젝트 루트: llm-data-analysis-course
원본 데이터 폴더: practice/chapter05/data/raw | 존재: True
결과 폴더: 공식 스크립트 → data/processed | 내 규칙 → data/processed/ch05_notebook


In [2]:
# 원본 4개 로드 + 원본 파일 해시 기록 (마지막에 원본이 바뀌지 않았는지 다시 확인)
FILES = ['customers', 'products', 'orders', 'order_items']
KEYS = {'customers': 'customer_id', 'products': 'product_id', 'orders': 'order_id', 'order_items': 'order_item_id'}


def file_md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


raw_hash_before = {name: file_md5(RAW_DIR / f'{name}.csv') for name in FILES}
raw = {name: pd.read_csv(RAW_DIR / f'{name}.csv') for name in FILES}
customers, products, orders, order_items = (raw[n] for n in FILES)
for name, df in raw.items():
    print(f'{name:12s} {str(df.shape):10s} {df.columns.tolist()}')

customers    (152, 6)   ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products     (102, 4)   ['product_id', 'product_name', 'category', 'price']
orders       (302, 5)   ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items  (765, 5)   ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [3]:
# [전처리 전 상태] shape · 결측 · 전체 중복 · PK 중복 · 관계용 ID의 자연스러운 반복
before_summary = pd.DataFrame([
    {'dataset': name, 'rows': len(df), 'columns': df.shape[1], 'missing_cells': int(df.isna().sum().sum()),
     'duplicated_rows': int(df.duplicated().sum()), 'pk': KEYS[name], 'pk_duplicated': int(df[KEYS[name]].duplicated().sum())}
    for name, df in raw.items()
])
display(before_summary)

missing_before = pd.concat({name: df.isna().sum() for name, df in raw.items()}, names=['dataset', 'column']).rename('missing').reset_index()
print('[결측이 있는 컬럼]')
display(missing_before[missing_before['missing'] > 0])

print('[관계용 ID 반복 — 중복 오류가 아님]')
print('order_items.order_id 반복 행:', int(order_items['order_id'].duplicated().sum()),
      '| 주문당 상품 수 범위:', order_items.groupby('order_id').size().min(), '~', order_items.groupby('order_id').size().max())
print('orders.customer_id 반복 행:', int(orders['customer_id'].duplicated().sum()))

,dataset,rows,columns,missing_cells,duplicated_rows,pk,pk_duplicated
0,customers,152,6,6,2,customer_id,2
1,products,102,4,0,2,product_id,2
2,orders,302,5,0,2,order_id,2
3,order_items,765,5,0,2,order_item_id,2


[결측이 있는 컬럼]


,dataset,column,missing
3,customers,age,4
4,customers,city,2


[관계용 ID 반복 — 중복 오류가 아님]
order_items.order_id 반복 행: 463 | 주문당 상품 수 범위: 1 ~ 4
orders.customer_id 반복 행: 174


In [4]:
# [타입 문제 후보] 숫자·날짜여야 하는데 문자열로 읽힌 컬럼과, 그대로 변환하면 실패하는 값
type_targets = [('customers', 'age', 'number'), ('products', 'price', 'number'),
                ('order_items', 'quantity', 'number'), ('order_items', 'unit_price', 'number'),
                ('customers', 'signup_date', 'date'), ('orders', 'order_date', 'date')]
rows = []
for name, col, kind in type_targets:
    s = raw[name][col]
    conv = pd.to_numeric(s, errors='coerce') if kind == 'number' else pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    failed = s.notna() & conv.isna()
    rows.append({'dataset': name, 'column': col, 'expected': kind, 'raw_dtype': str(s.dtype), 'non_null': int(s.notna().sum()),
                 'naive_convert_fail': int(failed.sum()), 'fail_values': sorted(s[failed].astype(str).unique().tolist())})
type_issues_before = pd.DataFrame(rows)
display(type_issues_before)

# [범주 실제 값] 고유값 수, 공백 포함 값, order_status·category 원본 값과 건수(repr로 공백까지 표시)
cat_targets = [('customers', 'city'), ('customers', 'gender'), ('products', 'category'),
               ('orders', 'payment_method'), ('orders', 'order_status')]
display(pd.DataFrame([{'dataset': n, 'column': c, 'unique_values': raw[n][c].nunique(),
                       'values_with_space': int(raw[n][c].dropna().astype(str).str.contains(r'\s').sum())} for n, c in cat_targets]))
for n, c in [('orders', 'order_status'), ('products', 'category')]:
    vc = raw[n][c].map(repr).value_counts()
    print(f'{n}.{c} ({len(vc)}종):', {k: int(v) for k, v in vc.items()})

# [전처리 전 PK/FK 연결 상태]
raw_fk = pd.DataFrame([
    {'relation': f'{ch}.{ck} → {pa}.{pk}', 'missing_in_parent': int((~raw[ch][ck].isin(raw[pa][pk])).sum()),
     'missing_ids': sorted(raw[ch].loc[~raw[ch][ck].isin(raw[pa][pk]), ck].unique().tolist())}
    for ch, ck, pa, pk in [('orders', 'customer_id', 'customers', 'customer_id'),
                           ('order_items', 'order_id', 'orders', 'order_id'),
                           ('order_items', 'product_id', 'products', 'product_id')]
])
display(raw_fk)

,dataset,column,expected,raw_dtype,non_null,naive_convert_fail,fail_values
0,customers,age,number,str,148,2,[unknown]
1,products,price,number,str,102,6,"[181,000, 28,000, 75,000, 93,000, unknown]"
2,order_items,quantity,number,str,765,5,"[2개, 3개, 5개, three, unknown]"
3,order_items,unit_price,number,str,765,7,"[129,000, 161,000, 17,000, 34,000, 52,000, not-a-number, unknown]"
4,customers,signup_date,date,str,152,2,"[2026-13-40, not-a-date]"
5,orders,order_date,date,str,302,3,"[2026-02-30, 2026/99/99, not-a-date]"


,dataset,column,unique_values,values_with_space
0,customers,city,15,5
1,customers,gender,2,0
2,products,category,13,6
3,orders,payment_method,9,5
4,orders,order_status,15,0


orders.order_status (15종): {"'completed'": 175, "'cancelled'": 59, "'refunded'": 53, "'Complete'": 2, "'cancel'": 2, "'Refund'": 2, "'complete'": 1, "'COMPLETED'": 1, "'완료'": 1, "'Cancel'": 1, "'CANCELLED'": 1, "'취소'": 1, "'refund'": 1, "'REFUNDED'": 1, "'환불'": 1}
products.category (13종): {"'뷰티'": 17, "'식품'": 17, "'패션'": 16, "'도서'": 15, "'전자기기'": 12, "'스포츠'": 11, "'생활용품'": 8, "'전자기기 '": 1, "'전자 기기'": 1, "' 생활용품'": 1, "'생활 용품'": 1, "'패션 '": 1, "'뷰 티'": 1}


,relation,missing_in_parent,missing_ids
0,orders.customer_id → customers.customer_id,2,"[9991, 9992]"
1,order_items.order_id → orders.order_id,2,"[9991, 9992]"
2,order_items.product_id → products.product_id,2,"[9991, 9992]"


- 각 데이터 shape: `customers` (152, 6), `products` (102, 4), `orders` (302, 5), `order_items` (765, 5)
- 결측치: `customers.age` 4, `customers.city` 2 (나머지 0)
- 전체 중복: 4개 테이블 모두 2행씩
- 주요 ID 중복: `customer_id` 2, `product_id` 2, `order_id` 2, `order_item_id` 2
- 타입 문제 후보: 숫자여야 하는 `age`, `price`, `quantity`, `unit_price`와 날짜여야 하는 `signup_date`, `order_date`가 모두 문자열(`str`)로 읽힘
- `order_status` 실제 값 15종: `completed` 175, `cancelled` 59, `refunded` 53 외에 `Complete`·`cancel`·`Refund` 각 2, `complete`·`COMPLETED`·`완료`·`Cancel`·`CANCELLED`·`취소`·`refund`·`REFUNDED`·`환불` 각 1
- `category` 실제 값 13종: 정상 7종 외에 `'전자기기 '`, `'전자 기기'`, `' 생활용품'`, `'생활 용품'`, `'패션 '`, `'뷰 티'` 각 1
- 전처리 전 PK/FK: 세 관계(`orders→customers`, `order_items→orders`, `order_items→products`) 모두 부모에 없는 ID가 2건씩이고, 그 ID는 모두 9991, 9992

![전처리 전 상태](images/step01_before.png)

### 결과 관찰
- 그대로 숫자·날짜로 변환하면 실패하는 값은 세 종류입니다.
  - **형식 문제**: `75,000` 같은 쉼표, `3개` 같은 단위 접미사
  - **의미 불명**: `three`, `unknown`, `not-a-number`
  - **존재하지 않는 날짜**: `2026-02-30`, `2026-13-40`, `2026/99/99`, `not-a-date`
- 범주 표기가 흩어져 있습니다. `order_status`는 15종(정상 3종), `category`는 13종(정상 7종)이고, `city`(15종)와 `payment_method`(9종)에는 앞뒤 공백이 섞여 있습니다.
- `order_items.order_id`는 463행이 반복되고(주문당 상품 1~4개), `orders.customer_id`는 174행이 반복됩니다. 이것은 1:N 관계에서 생기는 **자연스러운 반복**이라 중복 오류가 아닙니다. 중복 오류 후보는 PK 중복 2건씩뿐입니다.

### 나의 해석과 판단
처리 순서는 다음과 같이 정했습니다.
1. **문자열 공백 정리**: 공백 때문에 같은 값이 다른 값으로 보이면 중복 판정과 범주 표준화가 틀어집니다.
2. **타입 변환 + 실패 확인**: 숫자·날짜가 되어야 범위 검사와 파생 컬럼 계산을 할 수 있습니다.
3. **결측 처리**: 변환 실패로 새로 생긴 결측까지 포함해 판단해야 합니다. 원래 결측(4)만 보면 `unknown`(2)을 놓칩니다.
4. **중복 → 범주 → 이상값 → 관계 재검증**

중복은 PK가 같은 행이 "완전히 같은 행"인지 먼저 확인한 뒤에만 지웁니다(3번). 관계용 ID 반복(`order_items.order_id` 463행)은 지우면 안 됩니다.

### 업무·분석적 의미
- 전처리 전 상태를 숫자로 남겨야 "몇 건을 왜 바꿨는가"를 설명할 수 있습니다. 이 기록이 있으면 Chapter 06 EDA 결과가 이상할 때 원인을 추적할 수 있습니다.

### 한계와 추가 확인 사항
- `isna()`는 `unknown`, `not-a-date` 같은 **문자로 된 결측**을 세지 않습니다. 그래서 처리 전 결측 6건은 실제보다 적게 보이는 숫자입니다. 타입 변환 후 다시 셌습니다(5번).
- 원천 시스템 정보가 없어서 `unknown`이 "미입력"인지 "고객이 응답 거부"인지 구분할 수 없습니다.

## 2. 문자열·타입·결측 처리

In [5]:
change_log = []   # 규칙별 영향 건수 기록


def log(dataset, step, rule, affected):
    change_log.append({'dataset': dataset, 'step': step, 'rule': rule, 'affected': int(affected)})


# [규칙 2-1] 문자열 앞뒤 공백 제거, 빈 문자열 → 결측
def strip_strings(df, dataset):
    result = df.copy()
    for col in result.columns:
        if pd.api.types.is_string_dtype(result[col]) or result[col].dtype == object:
            before = result[col]
            after = before.where(before.isna(), before.astype(str).str.strip()).replace('', np.nan)
            changed = (before.astype(str) != after.astype(str)) & before.notna()
            if changed.sum():
                log(dataset, '2-문자열', f'{col} 앞뒤 공백 제거', changed.sum())
            result[col] = after
    return result


clean = {name: strip_strings(raw[name], name) for name in FILES}   # raw는 그대로 두고 복사본에서 처리

display(pd.DataFrame(change_log))
# 문자열 정리 후 결측 재확인 (빈 문자열이 결측으로 바뀌었는지)
print('문자열 정리 후 결측 셀 수:', {n: int(df.isna().sum().sum()) for n, df in clean.items()},
      '| 정리 전:', {n: int(df.isna().sum().sum()) for n, df in raw.items()})

,dataset,step,rule,affected
0,customers,2-문자열,city 앞뒤 공백 제거,5
1,products,2-문자열,category 앞뒤 공백 제거,3
2,orders,2-문자열,payment_method 앞뒤 공백 제거,5


문자열 정리 후 결측 셀 수: {'customers': 6, 'products': 0, 'orders': 0, 'order_items': 0} | 정리 전: {'customers': 6, 'products': 0, 'orders': 0, 'order_items': 0}


In [6]:
# [규칙 2-2] 숫자 변환: 쉼표 제거, quantity는 단위 접미사 '개'만 제거. 의미를 추측해야 하는 문자(three, unknown 등)는 NaN
def parse_number(series, suffix=None):
    text = series.astype(str).str.replace(',', '', regex=False)
    if suffix:
        text = text.str.replace(rf'{suffix}$', '', regex=True)
    return pd.to_numeric(text.where(series.notna()), errors='coerce')


num_rows = []
for name, col, suffix in [('customers', 'age', None), ('products', 'price', None),
                          ('order_items', 'quantity', '개'), ('order_items', 'unit_price', None)]:
    df = clean[name]
    original = df[col].copy()
    converted = parse_number(original, suffix)
    recovered = pd.to_numeric(original, errors='coerce').isna() & converted.notna() & original.notna()
    failed = original.notna() & converted.isna()
    df[col + '_raw'] = original.where(failed)          # 변환 실패 원문은 별도 컬럼에 보존
    df[col] = converted
    log(name, '2-숫자', f'{col} 형식 정리로 복구(쉼표/접미사)', recovered.sum())
    log(name, '2-숫자', f'{col} 변환 실패 → NaN', failed.sum())
    num_rows.append({'dataset': name, 'column': col, 'recovered': int(recovered.sum()),
                     'recovered_examples': sorted(original[recovered].astype(str).unique().tolist()),
                     'failed': int(failed.sum()), 'failed_values': sorted(original[failed].astype(str).unique().tolist()),
                     'missing_before': int(original.isna().sum()), 'nan_after': int(converted.isna().sum())})
number_conversion = pd.DataFrame(num_rows)

display(number_conversion)

,dataset,column,recovered,recovered_examples,failed,failed_values,missing_before,nan_after
0,customers,age,0,[],2,[unknown],4,6
1,products,price,4,"[181,000, 28,000, 75,000, 93,000]",2,[unknown],0,2
2,order_items,quantity,3,"[2개, 3개, 5개]",2,"[three, unknown]",0,2
3,order_items,unit_price,5,"[129,000, 161,000, 17,000, 34,000, 52,000]",2,"[not-a-number, unknown]",0,2


In [7]:
# [규칙 2-3] 날짜 변환: 형식을 YYYY-MM-DD로 고정, 실패는 NaT로 남김(임의 대체 안 함)
date_rows = []
for name, col in [('customers', 'signup_date'), ('orders', 'order_date')]:
    df = clean[name]
    original = df[col].copy()
    converted = pd.to_datetime(original, format='%Y-%m-%d', errors='coerce')
    failed = original.notna() & converted.isna()
    df[col + '_raw'] = original.where(failed)
    df[col] = converted
    log(name, '2-날짜', f'{col} 변환 실패 → NaT', failed.sum())
    date_rows.append({'dataset': name, 'column': col, 'converted': int(converted.notna().sum()), 'failed': int(failed.sum()),
                      'failed_values': sorted(original[failed].astype(str).unique().tolist()),
                      'min': converted.min().date(), 'max': converted.max().date()})
date_conversion = pd.DataFrame(date_rows)

display(date_conversion)
o_dt = clean['orders']['order_date']
print('order_date 변환 성공(중복 제거 전 302행 기준)', int(o_dt.notna().sum()), '건 중 작성일(2026-10-15) 이후', int((o_dt > pd.Timestamp('2026-10-15')).sum()), '건',
      '→ 분석 가능한 주문일 범위:', o_dt.min().date(), '~', o_dt[o_dt <= pd.Timestamp('2026-10-15')].max().date())

,dataset,column,converted,failed,failed_values,min,max
0,customers,signup_date,150,2,"[2026-13-40, not-a-date]",2023-07-21,2026-08-24
1,orders,order_date,299,3,"[2026-02-30, 2026/99/99, not-a-date]",2025-07-01,2028-12-31


order_date 변환 성공(중복 제거 전 302행 기준) 299 건 중 작성일(2026-10-15) 이후 2 건 → 분석 가능한 주문일 범위: 2025-07-01 ~ 2026-08-30


In [8]:
# [규칙 2-4] 결측 처리: age는 중앙값 대체 + 플래그, city는 Unknown, 날짜·가격·수량·단가는 대체하지 않음
AGE_VALID_MAX = 120
c = clean['customers']
age_valid = c['age'].where(c['age'].between(0, AGE_VALID_MAX))
age_median = age_valid.median()                       # 범위 밖 값(150)은 중앙값 계산에서 제외
c['age_imputed'] = c['age'].isna()
c['age'] = c['age'].fillna(age_median)
log('customers', '2-결측', f'age 결측(원래 결측+변환 실패) → 중앙값 {age_median} 대체, age_imputed 표시', c['age_imputed'].sum())
city_na = c['city'].isna()
c['city'] = c['city'].fillna('Unknown')
log('customers', '2-결측', 'city 결측 → Unknown', city_na.sum())

c = clean['customers']
print('age 유효값(0~120) 범위:', age_valid.min(), '~', age_valid.max(), '| 범위 밖:', c.loc[~c['age'].between(0, AGE_VALID_MAX), 'age'].tolist())
dedup_age = pd.to_numeric(raw['customers'].drop_duplicates()['age'], errors='coerce')
print(f"중앙값 계산 기준 확인 — 중복 포함 {len(age_valid)}행: {age_median} / 중복 제거 {len(dedup_age)}행: {dedup_age[dedup_age <= AGE_VALID_MAX].median()}")
missing_policy = pd.DataFrame([
    ['customers.age', '중앙값 대체 + age_imputed 플래그', int(c['age_imputed'].sum()), f'중앙값 {age_median}'],
    ['customers.city', "'Unknown' 범주로 유지", int(city_na.sum()), '지역 분석에서 별도 그룹'],
    ['customers.signup_date', 'NaT 유지(대체 안 함)', int(c['signup_date'].isna().sum()), '원문은 signup_date_raw에 보존'],
    ['orders.order_date', 'NaT 유지(대체 안 함), 월 집계 제외', int(clean['orders']['order_date'].isna().sum()), '원문은 order_date_raw에 보존'],
    ['products.price', 'NaN 유지(대체 안 함), 상품 행은 유지', int(clean['products']['price'].isna().sum()), '상품 주문 금액은 unit_price로 계산'],
    ['order_items.quantity', 'NaN 유지 → line_total 계산 제외', int(clean['order_items']['quantity'].isna().sum()), '원문은 quantity_raw에 보존'],
    ['order_items.unit_price', 'NaN 유지 → line_total 계산 제외', int(clean['order_items']['unit_price'].isna().sum()), '원문은 unit_price_raw에 보존'],
], columns=['column', '처리 기준', '영향 건수', '비고'])
display(missing_policy)

age 유효값(0~120) 범위: 18.0 ~ 90.0 | 범위 밖: [150.0]
중앙값 계산 기준 확인 — 중복 포함 152행: 41.0 / 중복 제거 150행: 41.0


,column,처리 기준,영향 건수,비고
0,customers.age,중앙값 대체 + age_imputed 플래그,6,중앙값 41.0
1,customers.city,'Unknown' 범주로 유지,2,지역 분석에서 별도 그룹
2,customers.signup_date,NaT 유지(대체 안 함),2,원문은 signup_date_raw에 보존
3,orders.order_date,"NaT 유지(대체 안 함), 월 집계 제외",3,원문은 order_date_raw에 보존
4,products.price,"NaN 유지(대체 안 함), 상품 행은 유지",2,상품 주문 금액은 unit_price로 계산
5,order_items.quantity,NaN 유지 → line_total 계산 제외,2,원문은 quantity_raw에 보존
6,order_items.unit_price,NaN 유지 → line_total 계산 제외,2,원문은 unit_price_raw에 보존


- 적용한 문자열 정리: 모든 문자열 컬럼의 앞뒤 공백 제거, 빈 문자열은 결측 처리
  - 변경 건수: `city` 5, `category` 3, `payment_method` 5
  - 정리 후 결측을 다시 세어 보니 6건으로 정리 전과 같았습니다. 빈 문자열은 없었습니다.
- 숫자 변환 대상과 실패 건수:
  | 컬럼 | 형식 정리로 복구 | 변환 실패 → NaN | 실패 원문 |
  | --- | ---: | ---: | --- |
  | `age` | 0 | 2 | `unknown` |
  | `price` | 4 (쉼표) | 2 | `unknown` |
  | `quantity` | 3 (`2개`, `3개`, `5개`) | 2 | `three`, `unknown` |
  | `unit_price` | 5 (쉼표) | 2 | `not-a-number`, `unknown` |
- 날짜 변환 대상과 실패 건수: 형식은 `%Y-%m-%d`로 고정했습니다.
  - `signup_date`: 성공 150 / 실패 2 (`2026-13-40`, `not-a-date`)
  - `order_date`: 성공 299 / 실패 3 (`2026-02-30`, `2026/99/99`, `not-a-date`) — 중복 제거 전 302행 기준
- 결측 처리 기준:
  - `age` 6건(원래 결측 4 + 변환 실패 2): 중앙값 41.0으로 대체하고 `age_imputed` 플래그를 붙였습니다.
  - `city` 2건: `Unknown`으로 처리했습니다.
  - 날짜·가격·수량·단가: **대체하지 않고** 결측으로 둡니다. 변환 실패 원문은 `*_raw` 컬럼에 보존합니다.

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
- 그대로 변환하면 실패하는 값 25건 중 12건(쉼표 9, `개` 3)은 형식만 정리해서 원래 숫자로 복구했습니다. 나머지 13건(숫자 8, 날짜 5)은 결측으로 남겼습니다.
- `order_date`는 변환에 성공한 299건 중 2건(2027-01-15, 2028-12-31)이 작성일(2026-10-15)보다 뒤입니다. 그래서 분석에 쓸 수 있는 주문일 범위는 **2025-07-01 ~ 2026-08-30**입니다.
- `age`의 유효값(0~120) 범위는 **18 ~ 90세**이고, 범위 밖 값은 150 하나입니다. 중앙값은 41.0입니다. 결측 대체를 중복 제거(3번)보다 먼저 했으므로 중앙값은 중복 포함 152행에서 계산됐습니다. 중복 제거 후 150행으로 계산해도 41.0으로 같다는 것을 출력으로 확인했습니다.

### 나의 해석과 판단
- **형식 문제는 고치고, 의미를 추측해야 하는 값은 고치지 않았습니다.**
  - `75,000` → 75000, `3개` → 3은 표기만 다를 뿐 값이 분명해서 복구했습니다.
  - `three`는 3일 가능성이 높습니다. 하지만 단어를 숫자로 바꾸는 규칙을 만들기 시작하면 `unknown`까지 추측하게 됩니다. 그래서 결측으로 두고 원문을 보존했습니다.
- **날짜는 대체하지 않았습니다.** `2026-02-30`을 02-28이나 03-02로 바꾸는 것은 근거 없는 추측입니다. 임의로 넣은 날짜는 월별 집계에 거짓 데이터를 섞습니다.
- **age는 중앙값으로 대체하되 플래그를 남겼습니다.**
  - 처리 전(중복 제거 후 150명) 평균 44.03은 150세 1건 때문에 150세를 뺀 평균 43.29보다 0.74세 높습니다. 중앙값은 150세를 넣든 빼든 41.0입니다(부록 LLM 검토에서 확인).
  - 그래서 이상값에 흔들리지 않는 중앙값을 골랐습니다.
- **정보 손실 가능성**
  - 중앙값 대체는 6명을 모두 41세에 몰아 넣어 분산을 줄입니다. 실제로 40대 인원이 24명에서 30명으로 늘었습니다(5번 민감도).
  - `city = Unknown`은 지역 분석에서 별도 그룹이 되어 지역 비중의 분모를 바꿉니다.
  - 형식 복구 규칙이 틀렸다면(예: `3개`가 사실 "3박스"였다면) 수량이 틀리게 됩니다.

### 업무·분석적 의미
- 같은 결측이라도 컬럼 역할에 따라 기준이 다릅니다. **식별자·금액·날짜는 대체 금지, 속성은 대체하되 표시**합니다. 이 기준을 남기면 다른 분석자도 같은 데이터로 같은 결과를 낼 수 있습니다.

### 한계와 추가 확인 사항
- `unknown`, `three` 원문이 어디서 생겼는지(입력 화면, 수기 입력, 외부 연동) 데이터에 정보가 없습니다. 원인을 알아야 처리 규칙을 확정할 수 있습니다.
- 연령대 분석에서는 `age_imputed` 제외 결과를 함께 보여 줘야 합니다. 40대 +6명 쏠림을 5번에서 숫자로 확인했습니다.

## 3. 중복·범주 표준화·이상값 후보

In [9]:
# [규칙 3-1] 중복: 문자열·타입·결측 처리를 마친 뒤 전체 행 중복을 다시 계산 → 완전 중복만 제거
dup_rows = []
for name in FILES:
    df, key = clean[name], KEYS[name]
    full_dup = df.duplicated()
    after = df.drop_duplicates().reset_index(drop=True)
    dup_rows.append({'dataset': name, 'full_duplicate_rows': int(full_dup.sum()),
                     'removed_keys': sorted(df.loc[full_dup, key].tolist()),
                     'key_dup_before': int(df[key].duplicated().sum()), 'key_dup_after': int(after[key].duplicated().sum())})
    log(name, '3-중복', '완전 중복 행 제거', full_dup.sum())
    clean[name] = after
duplicate_check = pd.DataFrame(dup_rows)

display(duplicate_check)

,dataset,full_duplicate_rows,removed_keys,key_dup_before,key_dup_after
0,customers,2,"[31, 76]",2,0
1,products,2,"[30, 84]",2,0
2,orders,2,"[44, 205]",2,0
3,order_items,2,"[109, 549]",2,0


In [10]:
# [규칙 3-2] 범주 표준화 + 허용값 검사
STATUS_MAP = {
    'complete': 'completed', 'Complete': 'completed', 'COMPLETED': 'completed', '완료': 'completed',
    'cancel': 'cancelled', 'Cancel': 'cancelled', 'CANCELLED': 'cancelled', '취소': 'cancelled',
    'refund': 'refunded', 'Refund': 'refunded', 'REFUNDED': 'refunded', '환불': 'refunded',
}
ALLOWED = {
    ('orders', 'order_status'): {'completed', 'cancelled', 'refunded'},
    ('orders', 'payment_method'): {'card', 'kakao_pay', 'naver_pay', 'bank_transfer'},
    ('products', 'category'): {'도서', '뷰티', '생활용품', '스포츠', '식품', '전자기기', '패션'},
    ('customers', 'city'): {'서울', '부산', '대구', '인천', '광주', '대전', '울산', '수원', '성남', '고양', 'Unknown'},
    ('customers', 'gender'): {'M', 'F'},
}


def outside(name, col):
    df = clean[name]
    return sorted(df.loc[~df[col].isin(ALLOWED[(name, col)]), col].astype(str).unique().tolist())


before_std = {k: outside(*k) for k in ALLOWED}
o = clean['orders']
status_changed = o['order_status'].isin(STATUS_MAP.keys())
o['order_status'] = o['order_status'].replace(STATUS_MAP)
log('orders', '3-범주', 'order_status 표기 통일', status_changed.sum())

p = clean['products']
cat_before = p['category'].copy()
p['category'] = p['category'].str.replace(r'\s+', '', regex=True)    # 내부 공백 제거('전자 기기' → '전자기기')
log('products', '3-범주', 'category 내부 공백 제거', (cat_before != p['category']).sum())

category_check = pd.DataFrame([
    {'column': f'{n}.{col}', 'outside_before': before_std[(n, col)], 'outside_after': outside(n, col)} for (n, col) in ALLOWED
])

display(category_check)
display(clean['orders']['order_status'].value_counts(dropna=False).to_frame('orders').T)

,column,outside_before,outside_after
0,orders.order_status,"[CANCELLED, COMPLETED, Cancel, Complete, REFUNDED, Refund, cancel, complete, refund, 완료, 취소, 환불]",[]
1,orders.payment_method,[],[]
2,products.category,"[뷰 티, 생활 용품, 전자 기기]",[]
3,customers.city,[],[]
4,customers.gender,[],[]


order_status,completed,cancelled,refunded
orders,180,63,57


In [11]:
# [규칙 3-3] 이상값 후보: 삭제·수정하지 않고 flag_* 컬럼으로 표시
AS_OF = pd.Timestamp('2026-10-15')     # 작성일. 이후 날짜는 '미래 주문일' 후보
c, p, o, oi = (clean[n] for n in FILES)
q1, q3 = oi['quantity'].quantile([0.25, 0.75])
q_upper = q3 + 1.5 * (q3 - q1)

c['flag_age_out_of_range'] = ~c['age'].between(0, AGE_VALID_MAX)
p['flag_price_invalid'] = p['price'].isna() | (p['price'] <= 0)
o['flag_date_invalid'] = o['order_date'].isna()
o['flag_future_date'] = o['order_date'] > AS_OF
oi['flag_qty_non_positive'] = oi['quantity'] <= 0
oi['flag_qty_large'] = oi['quantity'] > q_upper
oi['flag_price_non_positive'] = oi['unit_price'] <= 0
signup = o.merge(c[['customer_id', 'signup_date']], on='customer_id', how='left', validate='many_to_one')
o['flag_order_before_signup'] = (signup['order_date'] < signup['signup_date']).to_numpy()

outlier_rules = [
    ('customers', 'age > 120', 'flag_age_out_of_range', 'age', 'customer_id'),
    ('products', 'price 결측 또는 ≤ 0', 'flag_price_invalid', 'price', 'product_id'),
    ('orders', f'order_date > {AS_OF.date()} (미래)', 'flag_future_date', 'order_date', 'order_id'),
    ('orders', 'order_date < signup_date', 'flag_order_before_signup', None, 'order_id'),
    ('order_items', 'quantity ≤ 0', 'flag_qty_non_positive', 'quantity', 'order_item_id'),
    ('order_items', f'quantity > Q3+1.5·IQR ({q_upper:g})', 'flag_qty_large', 'quantity', 'order_item_id'),
    ('order_items', 'unit_price ≤ 0', 'flag_price_non_positive', 'unit_price', 'order_item_id'),
]
rows = []
for name, rule, flag, col, key in outlier_rules:
    df = clean[name]
    m = df[flag]
    ids = df.loc[m, key].tolist()
    vals = [] if col is None else [str(v.date()) if hasattr(v, 'date') else v for v in df.loc[m, col].tolist()]
    rows.append({'dataset': name, 'rule': rule, 'count': int(m.sum()),
                 'share(%)': round(m.mean() * 100, 1), 'values': vals[:6], 'ids': ids[:6] + (['…'] if len(ids) > 6 else [])})
    log(name, '3-이상값', f'후보 플래그: {rule}', m.sum())
outlier_candidates = pd.DataFrame(rows).assign(처리='삭제·수정 안 함, 플래그')

display(outlier_candidates)
big_orders = clean['order_items'].loc[clean['order_items']['flag_qty_large'], ['order_item_id', 'order_id', 'quantity', 'unit_price']]
big_info = (big_orders.merge(clean['orders'][['order_id', 'order_status']], on='order_id', how='left')
            .merge(clean['order_items'][['order_item_id', 'product_id']], on='order_item_id')
            .merge(clean['products'][['product_id', 'category']], on='product_id', how='left'))
print('대량 수량 후보 주문의 상태:', big_info[['order_item_id', 'order_id', 'quantity', 'order_status', 'category']].to_dict('records'))
_oi = clean['order_items'].merge(clean['orders'][['order_id', 'order_status']], on='order_id', how='left').merge(clean['products'][['product_id', 'category']], on='product_id', how='left')
for _, r in big_info[big_info['order_status'] == 'completed'].iterrows():
    cat_amt = (_oi.loc[(_oi['order_status'] == 'completed') & (_oi['category'] == r['category'])]
               .eval('quantity * unit_price').where(lambda x: x > 0).sum())
    row_amt = r['quantity'] * r['unit_price']
    print(f"completed 대량 수량 행 {r['order_item_id']}: {row_amt:,.0f} = '{r['category']}' 카테고리 completed 상품 주문 금액 {cat_amt:,.0f}의 {row_amt / cat_amt:.1%}")
_valid = _oi[(_oi['order_status'] == 'completed') & (_oi['quantity'] > 0) & (_oi['unit_price'] > 0)].assign(amount=lambda d: d['quantity'] * d['unit_price'])
_cat = pd.DataFrame({'포함': _valid.groupby('category')['amount'].sum(),
                     '제외(대량 수량 행)': _valid[~_valid['order_item_id'].isin(big_orders['order_item_id'])].groupby('category')['amount'].sum()})
_cat['순위_포함'] = _cat['포함'].rank(ascending=False).astype(int)
_cat['순위_제외'] = _cat['제외(대량 수량 행)'].rank(ascending=False).astype(int)
display(_cat.sort_values('포함', ascending=False))

,dataset,rule,count,share(%),values,ids,처리
0,customers,age > 120,1,0.7,[150.0],[33],"삭제·수정 안 함, 플래그"
1,products,price 결측 또는 ≤ 0,4,4.0,"[nan, 0.0, nan, -20000.0]","[18, 48, 73, 95]","삭제·수정 안 함, 플래그"
2,orders,order_date > 2026-10-15 (미래),2,0.7,"[2027-01-15, 2028-12-31]","[256, 287]","삭제·수정 안 함, 플래그"
3,orders,order_date < signup_date,58,19.3,[],"[3, 6, 7, 11, 15, 16, …]","삭제·수정 안 함, 플래그"
4,order_items,quantity ≤ 0,3,0.4,"[0.0, -1.0, 0.0]","[511, 612, 703]","삭제·수정 안 함, 플래그"
5,order_items,quantity > Q3+1.5·IQR (7),2,0.3,"[50.0, 100.0]","[738, 751]","삭제·수정 안 함, 플래그"
6,order_items,unit_price ≤ 0,1,0.1,[0.0],[579],"삭제·수정 안 함, 플래그"


대량 수량 후보 주문의 상태: [{'order_item_id': 738, 'order_id': 290, 'quantity': 50.0, 'order_status': 'completed', 'category': '패션'}, {'order_item_id': 751, 'order_id': 295, 'quantity': 100.0, 'order_status': 'refunded', 'category': '패션'}]
completed 대량 수량 행 738: 4,800,000 = '패션' 카테고리 completed 상품 주문 금액 34,200,000의 14.0%


,포함,제외(대량 수량 행),순위_포함,순위_제외
category,,,,
패션,34200000.0,29400000.0,1,1
식품,25421000.0,25421000.0,2,2
전자기기,21590000.0,21590000.0,3,3
도서,20003000.0,20003000.0,4,4
뷰티,17926000.0,17926000.0,5,5
스포츠,13401000.0,13401000.0,6,6
생활용품,9109000.0,9109000.0,7,7


- 제거/보류한 중복: 문자열·타입·결측 처리를 마친 뒤 다시 계산해도 완전 중복은 테이블마다 2행이었습니다.
  - 제거한 행: `customer_id` 31·76, `product_id` 30·84, `order_id` 44·205, `order_item_id` 109·549
  - 제거 후 PK 중복은 0이라 보류한 중복은 없습니다.
- 표준화한 범주값:
  - `order_status`: 15종 → 3종(15행 변경). 결과는 completed 180 / cancelled 63 / refunded 57
  - `category`: 앞뒤 공백 제거 3행 + 내부 공백 제거 3행(`뷰 티`→`뷰티`, `생활 용품`→`생활용품`, `전자 기기`→`전자기기`) → 13종 → 7종
- 허용값 밖 값: 표준화 후 `order_status`, `payment_method`, `category`, `city`, `gender` 모두 0건
- 이상값 후보: 모두 **삭제·수정하지 않고 `flag_*` 컬럼으로 표시**했습니다.

  | 대상 | 규칙 | 건수(비율) | 값 |
  | --- | --- | ---: | --- |
  | customers | age > 120 | 1 (0.7%) | 150 (customer 33) |
  | products | price 결측 또는 ≤ 0 | 4 (4.0%) | NaN×2, 0, -20,000 |
  | orders | order_date > 2026-10-15 | 2 (0.7%) | 2027-01-15, 2028-12-31 |
  | orders | order_date < signup_date | 58 (19.3%) | - |
  | order_items | quantity ≤ 0 | 3 (0.4%) | 0, -1, 0 |
  | order_items | quantity > Q3+1.5·IQR(=7) | 2 (0.3%) | 50(completed 주문), 100(refunded 주문) |
  | order_items | unit_price ≤ 0 | 1 (0.1%) | 0 |

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰
- 제거한 8행은 모두 모든 컬럼이 같은 완전 중복이었습니다. PK만 같고 내용이 다른 행은 없었습니다.
- `'전자기기 '`(끝 공백)는 앞뒤 공백 제거로 정리됐습니다. 하지만 `'전자 기기'`(가운데 공백)는 앞뒤 공백 제거로 정리되지 않아 별도 규칙이 필요했습니다.
- 대량 수량 후보 2행은 모두 패션 상품입니다. 수량 50(order 290)은 completed, 수량 100(order 295)은 refunded 주문입니다. completed인 수량 50 행(4,800,000)은 패션 카테고리 completed 상품 주문 금액(34,200,000)의 14.0%입니다. 이 행을 빼도 카테고리 순위(1~7위)는 그대로이고, 패션 1위와 2위 식품의 차이만 8,779,000 → 3,979,000으로 줄어듭니다.

### 나의 해석과 판단
이상값을 자동으로 삭제하지 않고 후보로 표시한 이유는, 같은 숫자라도 업무 의미에 따라 처리가 달라지기 때문입니다.
- **수량 100, 50**: 오입력일 수도, 실제 대량 주문일 수도 있습니다. 수량 100 주문은 refunded라서 completed 분석에서는 자동으로 빠집니다. 하지만 수량 50은 completed라 남습니다. 전체 금액에서는 3.4%(5번), 패션 카테고리 안에서는 14.0%입니다. 이 데이터에서는 빼도 순위가 바뀌지 않았지만 1·2위 격차가 절반 이하로 줄어듭니다. 지우면 실제 큰 거래가 사라지고, 두면 카테고리 격차가 과장될 수 있습니다.
- **수량 -1, 0**: 반품·취소 기록일 수 있습니다. 반품이면 금액에서 빼야 하므로 "오류라서 삭제"와 결과가 다릅니다.
- **가격 0원**: 증정품일 수 있습니다.
- **age 150, 미래 주문일**: 현실적으로 불가능한 값이라 오류일 가능성이 큽니다. 그래도 올바른 값을 알 근거가 없어서 값을 바꾸지 않고 표시만 했습니다.
- **주문일 < 가입일 58건(19.3%)**: 개별 오입력으로 보기에는 비율이 너무 높습니다. `signup_date`의 정의(최초 가입인지 재가입인지)나 생성 방식의 문제일 가능성이 큽니다. 그래서 행은 유지하고, 가입 후 경과 기간 분석에는 이 컬럼을 쓰지 않기로 했습니다.
- 삭제는 되돌릴 수 없지만 플래그는 분석 단계에서 포함/제외를 고를 수 있습니다. 그래서 **전처리에서는 판단 근거를 남기고, 제외 여부는 분석 질문에 맞춰 정합니다.**

### 업무·분석적 의미
- 이상값 후보 목록(`ids`)은 그대로 업무 담당자에게 보낼 확인 요청 목록이 됩니다. 예: "order_item 738은 실제 50개 주문인가?"
- 표준화 후 허용값 검사를 다시 한 덕분에, 매핑에서 빠진 값이 집계에서 조용히 별도 그룹이 되는 문제를 막았습니다.

### 한계와 추가 확인 사항
- IQR 상한(7)은 통계적 경계일 뿐 업무 기준이 아닙니다. 고객당 구매 한도 같은 업무 규칙이 있으면 그것으로 바꿔야 합니다.
- 미래 날짜 판단 기준일(2026-10-15)은 작성일입니다. 데이터 추출일 정보가 있으면 그 날짜를 기준으로 해야 합니다.

## 4. PK/FK와 파생 컬럼 재검증

In [12]:
# [규칙 4-1] 파생 컬럼: 날짜·수량·단가가 유효한 행에서만 계산
o, oi = clean['orders'], clean['order_items']
o['order_month'] = o['order_date'].dt.strftime('%Y-%m')          # NaT → NaN (문자열 'NaT'가 생기지 않게)
o['order_dayofweek'] = o['order_date'].dt.day_name()
valid_line = (oi['quantity'] > 0) & (oi['unit_price'] > 0)
oi['is_valid_line'] = valid_line
oi['line_total'] = (oi['quantity'] * oi['unit_price']).where(valid_line)

derived_check = pd.DataFrame([
    ['orders.order_month', int(o['order_month'].notna().sum()), int(o['order_month'].isna().sum()),
     f"문자열 'NaT' {int((o['order_month'].astype(str) == 'NaT').sum())}건, NaN 수 = order_date 결측 수: {o['order_month'].isna().sum() == o['order_date'].isna().sum()}"],
    ['orders.order_dayofweek', int(o['order_dayofweek'].notna().sum()), int(o['order_dayofweek'].isna().sum()), ''],
    ['order_items.line_total', int(oi['line_total'].notna().sum()), int(oi['line_total'].isna().sum()),
     f"유효 행에서 quantity×unit_price와 불일치 {int((oi.loc[valid_line, 'line_total'] != oi.loc[valid_line, 'quantity'] * oi.loc[valid_line, 'unit_price']).sum())}건"],
], columns=['derived_column', 'filled', 'NaN', 'check'])
display(derived_check)
print('유효 행 상품 주문 금액(quantity × unit_price) 합계, 주문 상태 구분 전:', f"{oi['line_total'].sum():,.0f}")

,derived_column,filled,NaN,check
0,orders.order_month,297,3,"문자열 'NaT' 0건, NaN 수 = order_date 결측 수: True"
1,orders.order_dayofweek,297,3,
2,order_items.line_total,755,8,유효 행에서 quantity×unit_price와 불일치 0건


유효 행 상품 주문 금액(quantity × unit_price) 합계, 주문 상태 구분 전: 253,289,000


In [13]:
# [규칙 4-2] PK/FK 재검증: 끊긴 관계는 삭제하지 않고 flag_fk_* 로 표시
fk_rows = []
for child, ck, parent, pk in [('orders', 'customer_id', 'customers', 'customer_id'),
                              ('order_items', 'order_id', 'orders', 'order_id'),
                              ('order_items', 'product_id', 'products', 'product_id')]:
    bad = ~clean[child][ck].isin(clean[parent][pk])
    clean[child][f'flag_fk_{ck}_missing'] = bad
    fk_rows.append({'check': f'{child}.{ck} → {parent}.{pk}', 'invalid_count': int(bad.sum()),
                    'missing_ids': sorted(clean[child].loc[bad, ck].unique().tolist()),
                    'child_rows': clean[child].loc[bad, KEYS[child]].tolist()})
    log(child, '4-관계', f'{ck} 참조 대상 없음 플래그', bad.sum())
fk_check = pd.DataFrame(fk_rows)
pk_check = pd.DataFrame([{'table': n, 'pk': KEYS[n], 'duplicated': int(df[KEYS[n]].duplicated().sum()),
                          'null': int(df[KEYS[n]].isna().sum())} for n, df in clean.items()])

display(pk_check)
display(fk_check)
print('정상 ID 최댓값(9991·9992 제외) — customer_id:', int(clean['customers']['customer_id'].max()),
      '| order_id:', int(clean['orders']['order_id'].max()), '| product_id:', int(clean['products']['product_id'].max()))

# 비교: 공식 규칙처럼 price가 잘못된 상품의 '행 전체를 삭제'했다면?
p, oi = clean['products'], clean['order_items']
bad_products = p.loc[p['flag_price_invalid'], 'product_id']
orphan_if_dropped = oi['product_id'].isin(bad_products)
print(f'가격 문제 상품 {len(bad_products)}개 {bad_products.tolist()}의 행을 삭제했다면 참조 대상이 없어지는 주문 상세: {int(orphan_if_dropped.sum())}행'
      f" (그중 수량·단가가 정상인 행 {int((orphan_if_dropped & oi['is_valid_line']).sum())}행)")

,table,pk,duplicated,null
0,customers,customer_id,0,0
1,products,product_id,0,0
2,orders,order_id,0,0
3,order_items,order_item_id,0,0


,check,invalid_count,missing_ids,child_rows
0,orders.customer_id → customers.customer_id,2,"[9991, 9992]","[126, 278]"
1,order_items.order_id → orders.order_id,2,"[9991, 9992]","[150, 620]"
2,order_items.product_id → products.product_id,2,"[9991, 9992]","[260, 701]"


정상 ID 최댓값(9991·9992 제외) — customer_id: 150 | order_id: 300 | product_id: 100
가격 문제 상품 4개 [18, 48, 73, 95]의 행을 삭제했다면 참조 대상이 없어지는 주문 상세: 24행 (그중 수량·단가가 정상인 행 24행)


- 없는 `customer_id`: 2건 (`orders` 126·278 → customer 9991·9992)
- 없는 `order_id`: 2건 (`order_items` 150·620 → order 9991·9992)
- 없는 `product_id`: 2건 (`order_items` 260·701 → product 9991·9992)
- 생성한 파생 컬럼:
  - `order_month`(YYYY-MM), `order_dayofweek`: 날짜 변환에 성공한 297건(중복 제거 후 300행 기준)에서만 만들고, 변환 실패 3건은 NaN. 미래 날짜 2건은 값은 만들되 `flag_future_date`로 구분
  - `line_total`: 수량·단가가 모두 양수인 755행에서만 계산하고, 8행은 NaN
  - `is_valid_line`
- `line_total` 검증:
  - 유효 행에서 `quantity × unit_price`와 불일치 0건
  - 유효 행 상품 주문 금액 합계 253,289,000 (주문 상태 구분 전, 회계상 매출 아님)
  - `order_month`에 문자열 `'NaT'` 0건, NaN 수가 `order_date` 결측 수(3)와 같음

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
- 전처리 후 4개 테이블 모두 PK 중복 0, PK 결측 0입니다.
- FK 누락은 관계마다 2건이고, 전처리 전(1번)과 같은 ID(9991, 9992)입니다. 즉 내 전처리는 관계를 하나도 새로 끊지 않았습니다.
- 비교 실험: 공식 규칙처럼 가격 문제 상품 4개(18·48·73·95)의 **행 전체를 삭제했다면**, 그 상품을 참조하는 주문 상세 **24행이 orphan이 됩니다**. 24행 모두 수량·단가가 정상인 거래입니다.

### 나의 해석과 판단
전처리 후 관계를 다시 검증해야 하는 이유는 **전처리에서 행을 지우면 다른 테이블의 참조가 끊기기 때문**입니다.
- 상품 마스터의 `price`는 정가 정보일 뿐이고, 상품 주문 금액은 `order_items.unit_price`로 계산합니다. 정가가 이상하다고 상품 행을 지우면 정상 거래 24건이 카테고리를 잃고 카테고리 집계에서 빠집니다.
- 그래서 마스터 테이블에서는 행을 지우지 않고 `flag_price_invalid`만 표시하는 것이 맞다고 판단했습니다.
- 정상 ID 최댓값은 customer 150, order 300, product 100입니다. 9991·9992는 이 범위를 크게 벗어나고 세 관계에 같은 번호로 나타납니다. 그래서 **테스트용 또는 연동 오류로 의심되는 ID**로 봤습니다. 확인된 사실은 아니라 삭제하지 않고 `flag_fk_*_missing`으로 표시해 분석에서 제외할 수 있게 했습니다.

### 업무·분석적 의미
- `order_month`를 `strftime`으로 만들고 NaN 수를 확인했기 때문에 월별 집계에 가짜 월(`'NaT'`)이 생기지 않습니다.
- `line_total`을 유효 행에서만 계산했기 때문에 "상품 주문 금액 합계"가 무엇을 포함하는지 명확합니다. 수량 ≤ 0, 단가 ≤ 0, 변환 실패 행은 제외됩니다.

## 5. 전처리 전/후 비교

In [14]:
# 전처리 전/후 비교
fail_before = type_issues_before.groupby('dataset')['naive_convert_fail'].sum()
compare_rows = []
for name in FILES:
    r, c = raw[name], clean[name]
    flags = [col for col in c.columns if col.startswith('flag_')]
    compare_rows.append({
        'dataset': name, 'rows_raw': len(r), 'rows_clean': len(c), 'cols_raw': r.shape[1], 'cols_clean': c.shape[1],
        'missing_raw(원 컬럼)': int(r.isna().sum().sum()), 'missing_clean(원 컬럼)': int(c[r.columns].isna().sum().sum()),
        'dup_rows_raw': int(r.duplicated().sum()), 'dup_rows_clean': int(c.duplicated().sum()),
        'convert_fail_raw': int(fail_before.get(name, 0)),
        'convert_fail_left(*_raw 보존)': int(sum(c[col].notna().sum() for col in c.columns if col.endswith('_raw'))),
        'flagged_rows': int(c[flags].any(axis=1).sum()) if flags else 0,
    })
before_after = pd.DataFrame(compare_rows).set_index('dataset')
display(before_after.T)

# 분포 변화: raw 기준은 '중복 제거 후 · 대체 전' (150명)으로 맞춤
age_before = pd.to_numeric(raw['customers'].drop_duplicates()['age'], errors='coerce')
age_after = clean['customers']['age']
dist = pd.DataFrame([
    ['customers.age 중앙값', age_before.median(), age_after.median(), age_after.median() - age_before.median()],
    ['customers.age 평균', round(age_before.mean(), 2), round(age_after.mean(), 2), round(age_after.mean() - age_before.mean(), 2)],
    ['order_status 종류 수', raw['orders']['order_status'].nunique(), clean['orders']['order_status'].nunique(), None],
    ['category 종류 수', raw['products']['category'].nunique(), clean['products']['category'].nunique(), None],
    ['city 종류 수', raw['customers']['city'].nunique(), clean['customers']['city'].nunique(), None],
], columns=['지표', '처리 전(중복 제거 후)', '처리 후', '변화'])
display(dist.astype(str))

dataset,customers,products,orders,order_items
rows_raw,152,102,302,765
rows_clean,150,100,300,763
cols_raw,6,4,5,5
cols_clean,10,6,12,14
missing_raw(원 컬럼),6,0,0,0
missing_clean(원 컬럼),2,2,3,4
dup_rows_raw,2,2,2,2
dup_rows_clean,0,0,0,0
convert_fail_raw,4,6,3,12
convert_fail_left(*_raw 보존),4,2,3,4


,지표,처리 전(중복 제거 후),처리 후,변화
0,customers.age 중앙값,41.0,41.0,0.0
1,customers.age 평균,44.03,43.91,-0.12
2,order_status 종류 수,15.0,3.0,NaN
3,category 종류 수,13.0,7.0,NaN
4,city 종류 수,15.0,11.0,NaN


In [15]:
# [민감도 확인] 처리 규칙이 다음 분석 결과를 얼마나 바꾸는가
c, oi = clean['customers'], clean['order_items']
bands = pd.cut(c['age'], bins=[0, 30, 40, 50, 60, 200], right=False, labels=['~20대', '30대', '40대', '50대', '60대+'])   # [30, 40) = 30대
age_band = pd.DataFrame({'대체 포함': bands.value_counts(), '대체 제외': bands[~c['age_imputed']].value_counts()}).sort_index()
age_band['차이'] = age_band['대체 포함'] - age_band['대체 제외']
display(age_band)
lt, big = oi['line_total'], oi['flag_qty_large']
print(f"유효 행 상품 주문 금액(상태 구분 전): 전체 {lt.sum():,.0f} / 대량 수량 후보 제외 {lt[~big].sum():,.0f} → 후보 2행 비중 {lt[big].sum() / lt.sum():.1%}")
done = oi.merge(clean['orders'][['order_id', 'order_status']], on='order_id', how='left')['order_status'].eq('completed').to_numpy()
print(f"completed만 보면: 전체 {lt[done].sum():,.0f} / 후보 제외 {lt[done & ~big].sum():,.0f} → 후보 비중 {lt[done & big].sum() / lt[done].sum():.1%}")

,대체 포함,대체 제외,차이
age,,,
~20대,35,35,0
30대,31,31,0
40대,30,24,6
50대,23,23,0
60대+,31,31,0


유효 행 상품 주문 금액(상태 구분 전): 전체 253,289,000 / 대량 수량 후보 제외 232,989,000 → 후보 2행 비중 8.0%
completed만 보면: 전체 141,726,000 / 후보 제외 136,926,000 → 후보 비중 3.4%


In [16]:
# [저장] 개인정보(name) 제외 + 변경 기록 저장
change_log_df = pd.DataFrame(change_log)
change_log_df = change_log_df[change_log_df['affected'] > 0].reset_index(drop=True)
to_save = {name: df.drop(columns=['name'], errors='ignore') for name, df in clean.items()}
OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, df in to_save.items():
    df.to_csv(OUT_DIR / f'{name}_clean.csv', index=False, encoding='utf-8-sig')
change_log_df.to_csv(OUT_DIR / 'change_log.csv', index=False, encoding='utf-8-sig')

reread_rows = []
for name, df in to_save.items():
    chk = pd.read_csv(OUT_DIR / f'{name}_clean.csv')
    date_cols = [col for col in ['order_date', 'signup_date'] if col in chk.columns]
    reread_rows.append({'file': f'{name}_clean.csv', 'shape': chk.shape, 'same_shape': chk.shape == df.shape,
                        'same_columns': chk.columns.tolist() == df.columns.tolist(), 'pk_dup': int(chk[KEYS[name]].duplicated().sum()),
                        'date_reparse_fail': {col: int((chk[col].notna() & pd.to_datetime(chk[col], errors='coerce').isna()).sum()) for col in date_cols},
                        'has_name_col': 'name' in chk.columns})
display(pd.DataFrame(reread_rows))
print('변경 기록:', len(change_log_df), '개 규칙')
display(change_log_df)

,file,shape,same_shape,same_columns,pk_dup,date_reparse_fail,has_name_col
0,customers_clean.csv,"(150, 9)",True,True,0,{'signup_date': 0},False
1,products_clean.csv,"(100, 6)",True,True,0,{},False
2,orders_clean.csv,"(300, 12)",True,True,0,{'order_date': 0},False
3,order_items_clean.csv,"(763, 14)",True,True,0,{},False


변경 기록: 30 개 규칙


,dataset,step,rule,affected
0,customers,2-문자열,city 앞뒤 공백 제거,5
1,products,2-문자열,category 앞뒤 공백 제거,3
2,orders,2-문자열,payment_method 앞뒤 공백 제거,5
3,customers,2-숫자,age 변환 실패 → NaN,2
4,products,2-숫자,price 형식 정리로 복구(쉼표/접미사),4
5,products,2-숫자,price 변환 실패 → NaN,2
6,order_items,2-숫자,quantity 형식 정리로 복구(쉼표/접미사),3
7,order_items,2-숫자,quantity 변환 실패 → NaN,2
8,order_items,2-숫자,unit_price 형식 정리로 복구(쉼표/접미사),5
9,order_items,2-숫자,unit_price 변환 실패 → NaN,2


| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 152 / 102 / 302 / 765 | 150 / 100 / 300 / 763 | 각 테이블 완전 중복 2행 제거. 그 외 삭제 없음 |
| 결측(원 컬럼) | 6 / 0 / 0 / 0 | 2 / 2 / 3 / 4 | age 6·city 2는 처리됨. 변환 실패가 결측으로 드러남(signup 2, price 2, order_date 3, quantity 2, unit_price 2) |
| 중복 | 2 / 2 / 2 / 2 | 0 / 0 / 0 / 0 | 완전 중복 제거 |
| 변환 실패 | 4 / 6 / 3 / 12 | 4 / 2 / 3 / 4 (원문 보존) | 쉼표·`개` 12건 복구, 남은 값은 `*_raw`에 원문 보존 |

(순서: customers / products / orders / order_items)

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰
- 행이 줄어든 원인은 **완전 중복 제거 하나뿐**입니다(총 8행).
- 컬럼은 늘었습니다: 플래그, 원문 보존 `*_raw`, 파생 컬럼(`order_month`, `order_dayofweek`, `line_total`, `is_valid_line`)
- 결측은 6 → 11로 늘었습니다. 문자로 숨어 있던 결측(`unknown`, 존재하지 않는 날짜)이 타입 변환 후 결측으로 드러났기 때문입니다.
- 범주 종류 수: `order_status` 15 → 3, `category` 13 → 7, `city` 15 → 11(10개 도시 + Unknown)
- age(처리 전 기준은 중복 제거 후 150명): 평균 44.03 → 43.91(−0.12), 중앙값 41.0 → 41.0
- 플래그가 붙은 행: customers 1, products 4, orders 65, order_items 10
- 민감도:
  - 대체 포함 시 40대 30명, 제외 시 24명입니다. 다른 연령대는 차이가 0입니다.
  - 대량 수량 후보 2행의 비중은 유효 행 상품 주문 금액(상태 구분 전 253,289,000)의 8.0%입니다. completed만 보면 141,726,000 중 3.4%입니다.

### 나의 해석과 판단
- 행 수 변화가 의도한 결과인지는 두 가지로 확인했습니다.
  - 줄어든 행 수(테이블당 2)가 3번에서 찾은 완전 중복 수와 같습니다.
  - 변경 기록(`change_log.csv`, 30개 규칙)의 영향 건수로 모든 변경을 설명할 수 있습니다.
- 결측 증가는 품질이 나빠진 것이 아니라 **보이지 않던 문제가 보이게 된 것**이라 의도한 결과입니다.
- 중앙값 대체로 평균은 0.12세만 바뀌어 전체 분포는 거의 그대로입니다. 그러나 연령대 단위에서는 40대만 25% 커집니다. **전체 통계가 괜찮아도 구간 분석에서는 영향이 있다**는 것을 숫자로 확인했습니다.
- 대량 수량 후보의 영향은 전체 completed 금액 기준으로는 3.4%로 작습니다. 패션 카테고리 안에서는 14.0%이고, 빼도 카테고리 순위는 그대로지만 패션과 식품의 격차가 8,779,000 → 3,979,000으로 줄어듭니다(3번). 그래서 행을 지우지 않고 플래그로 포함/제외 결과를 모두 볼 수 있게 했습니다.

### 업무·분석적 의미
- "결측이 줄었다 = 좋아졌다"가 아닙니다. 결측 수 변화는 **이유와 함께** 기록해야 오해가 없습니다.
- 플래그 열 덕분에 Chapter 06에서 "전체 / 플래그 제외" 두 버전의 결과를 쉽게 비교할 수 있습니다.

### 한계와 추가 확인 사항
- `orders` 플래그 65행 중 58행은 "주문일 < 가입일"입니다. 이 규칙의 원인이 확인되기 전까지 orders 플래그 수는 실제 오류보다 많게 보입니다.
- 이 비교표는 행·결측·중복 수준입니다. 값이 바뀐 셀의 상세 내역은 `change_log.csv`와 `*_raw` 컬럼으로 확인합니다.

### 저장 결과 재확인
- 내 규칙 결과는 `data/processed/ch05_notebook/`에 저장했습니다. 다시 읽었을 때 shape, 컬럼, PK 고유성이 유지되고 날짜 재변환 실패는 0건입니다.
- `name`은 저장하지 않았습니다(`has_name_col` False).

## 6. 재현 가능한 전처리 확인

In [17]:
# (1) 공식 스크립트 재실행: ① 안내 명령 그대로 ② 프로젝트 루트를 import 경로에 추가 ③ 같은 명령 재실행 후 해시 비교
OFFICIAL = ['scripts/preprocess_data.py']
OFFICIAL_OUT = [PROCESSED_DIR / f'{n}_clean.csv' for n in FILES] + [REPORT_DIR / 'ch05_preprocessing_summary.md']


def run(cmd, extra_env=None, cwd=PROJECT_ROOT):
    env = {**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHON_COLORS': '0', 'NO_COLOR': '1', **(extra_env or {})}
    return subprocess.run([sys.executable, *cmd], cwd=cwd, env=env, capture_output=True, text=True, encoding='utf-8')


existing_processed = {n: pd.read_csv(PROCESSED_DIR / f'{n}_clean.csv') for n in FILES if (PROCESSED_DIR / f'{n}_clean.csv').exists()}
r1 = run(OFFICIAL)
r2 = run(OFFICIAL, {'PYTHONPATH': str(PROJECT_ROOT)}); h2 = {p.name: file_md5(p) for p in OFFICIAL_OUT}
r3 = run(OFFICIAL, {'PYTHONPATH': str(PROJECT_ROOT)}); h3 = {p.name: file_md5(p) for p in OFFICIAL_OUT}
print('① python scripts/preprocess_data.py → returncode', r1.returncode, '|', (r1.stderr.strip().splitlines() or [''])[-1])
print('② PYTHONPATH=<프로젝트 루트> → returncode', r2.returncode, '| ③ 재실행 → returncode', r3.returncode)
display(pd.DataFrame({'file': list(h2), 'run2_md5': list(h2.values()), 'run3_md5': list(h3.values())}).assign(identical=lambda d: d['run2_md5'] == d['run3_md5']))
same = {}
for n, old in existing_processed.items():
    try:
        pd.testing.assert_frame_equal(old, pd.read_csv(PROCESSED_DIR / f'{n}_clean.csv'), check_dtype=False); same[n] = True
    except AssertionError:
        same[n] = False
print('실행 전 data/processed에 있던 파일과 내용 동일:', same if existing_processed else '(실행 전 파일 없음)')
print('\n[② 실행 출력]')
print('\n'.join(l for l in r2.stdout.splitlines() if l.strip()))

① python scripts/preprocess_data.py → returncode 1 | ModuleNotFoundError: No module named 'src'
② PYTHONPATH=<프로젝트 루트> → returncode 0 | ③ 재실행 → returncode 0


,file,run2_md5,run3_md5,identical
0,customers_clean.csv,ab0f7249fa9d43ec8ccfe8753407c46a,ab0f7249fa9d43ec8ccfe8753407c46a,True
1,products_clean.csv,722dc82d080f5981ea2ed008d070d809,722dc82d080f5981ea2ed008d070d809,True
2,orders_clean.csv,151ee68c2727be5db11186e360482d43,151ee68c2727be5db11186e360482d43,True
3,order_items_clean.csv,33974675fcb43abb2c184086aa928f2b,33974675fcb43abb2c184086aa928f2b,True
4,ch05_preprocessing_summary.md,bf61952eb56f8898baf79d97b2530fd6,bf61952eb56f8898baf79d97b2530fd6,True


실행 전 data/processed에 있던 파일과 내용 동일: {'customers': True, 'products': True, 'orders': True, 'order_items': True}

[② 실행 출력]
전처리 완료
입력 Raw 경로: practice/chapter05/data/raw
[전처리 전후 데이터 크기]
    dataset  rows_raw  columns_raw  rows_processed  columns_processed
  customers       152            6             150                  6
order_items       765            5             752                  6
     orders       302            5             300                  7
   products       102            4              96                  4
[중복 점검 결과]
    dataset  row_duplicate_count    key_column  key_duplicate_count
  customers                    0   customer_id                    0
   products                    0    product_id                    0
     orders                    0      order_id                    0
order_items                    0 order_item_id                    0
[파일 간 관계 점검 결과]
                                               check  invalid_count
  orders.customer_id exists in c

In [18]:
# (2) 내 규칙 재현: Notebook 규칙 셀과 같은 코드로 만든 ch05_preprocess.py를 2회 실행 → Notebook 결과와 바이트 단위 비교
MY_SCRIPT = Path.cwd() / 'ch05_preprocess.py'
print('실행한 스크립트:', MY_SCRIPT.name, '| MD5:', file_md5(MY_SCRIPT), '(제출 파일 chapter05/ch05_preprocess.py와 같은 파일)')
check_dirs = [PROCESSED_DIR / 'ch05_script_run1', PROCESSED_DIR / 'ch05_script_run2']
my_rows = []
for d in check_dirs:
    r = run([str(MY_SCRIPT), str(d)])
    print(MY_SCRIPT.name, '→ returncode', r.returncode, '|', r.stdout.strip() or r.stderr.strip().splitlines()[-1])
for f in [f'{n}_clean.csv' for n in FILES] + ['change_log.csv']:
    hashes = [file_md5(OUT_DIR / f)] + [file_md5(d / f) for d in check_dirs]
    my_rows.append({'file': f, 'notebook_md5': hashes[0], 'script_run1': hashes[1] == hashes[0], 'script_run2': hashes[2] == hashes[0]})
display(pd.DataFrame(my_rows))

실행한 스크립트: ch05_preprocess.py | MD5: 57bb6e0df42d9051630a7e679d78cc43 (제출 파일 chapter05/ch05_preprocess.py와 같은 파일)


ch05_preprocess.py → returncode 0 | checks: 허용값 밖 0 | PK 중복·결측 0 | raw 변경 없음 | FK 누락 {'orders.customer_id → customers.customer_id': 2, 'order_items.order_id → orders.order_id': 2, 'order_items.product_id → products.product_id': 2}
saved: data/processed/ch05_script_run1 | rows: {'customers': 150, 'products': 100, 'orders': 300, 'order_items': 763} | change_log: 30


ch05_preprocess.py → returncode 0 | checks: 허용값 밖 0 | PK 중복·결측 0 | raw 변경 없음 | FK 누락 {'orders.customer_id → customers.customer_id': 2, 'order_items.order_id → orders.order_id': 2, 'order_items.product_id → products.product_id': 2}
saved: data/processed/ch05_script_run2 | rows: {'customers': 150, 'products': 100, 'orders': 300, 'order_items': 763} | change_log: 30


,file,notebook_md5,script_run1,script_run2
0,customers_clean.csv,6c417964619b4c8b601bc4df76201573,True,True
1,products_clean.csv,7c731a09ff20bc90f7d4289a012eb40d,True,True
2,orders_clean.csv,87b801569e65d266bb4447b6295268fe,True,True
3,order_items_clean.csv,f5a95bb86bc4f998320a4c9e421fe136,True,True
4,change_log.csv,f93c43fe00b0d82dfa08d02f1cb9461b,True,True


In [19]:
# (3) 공식 스크립트 결과 vs 내 규칙 결과 + 저장 결과 재확인
script_out = {n: pd.read_csv(PROCESSED_DIR / f'{n}_clean.csv') for n in FILES}
display(pd.DataFrame([{'dataset': n, 'rows_script': len(script_out[n]), 'rows_mine': len(clean[n]),
                       'cols_script': script_out[n].shape[1], 'cols_mine': clean[n].shape[1],
                       'script_columns': script_out[n].columns.tolist()} for n in FILES]))
so = script_out['orders']
print('공식 결과 order_date 재변환 실패:', int((so['order_date'].notna() & pd.to_datetime(so['order_date'], errors='coerce').isna()).sum()),
      '| order_date 결측:', int(so['order_date'].isna().sum()))
print('공식 결과 category 값:', sorted(script_out['products']['category'].unique().tolist()))
print('공식 스크립트가 삭제한 product_id:', sorted(set(clean['products']['product_id']) - set(script_out['products']['product_id'])))
print('공식 결과 product FK 누락:', int((~script_out['order_items']['product_id'].isin(script_out['products']['product_id'])).sum()),
      '/ 내 규칙:', int(clean['order_items']['flag_fk_product_id_missing'].sum()))
dropped = raw['order_items'].drop_duplicates()
dropped = dropped[~dropped['order_item_id'].isin(script_out['order_items']['order_item_id'])][['order_item_id', 'quantity', 'unit_price']]
dropped['사유'] = np.select(
    [dropped['quantity'].str.endswith('개'), pd.to_numeric(dropped['quantity'], errors='coerce').isna() | pd.to_numeric(dropped['unit_price'], errors='coerce').isna()],
    ['형식 문제(개)', '변환 실패'], '0 이하 값')
print('공식 스크립트가 삭제한 주문 상세(중복 제외):', dropped['사유'].value_counts().to_dict())
print('공식 결과 age 최댓값:', script_out['customers']['age'].max(), '| 미래 order_date 포함:', int((pd.to_datetime(so['order_date']) > AS_OF).sum()))

,dataset,rows_script,rows_mine,cols_script,cols_mine,script_columns
0,customers,150,150,6,10,"[customer_id, name, gender, age, city, signup_date]"
1,products,96,100,4,6,"[product_id, product_name, category, price]"
2,orders,300,300,7,12,"[order_id, customer_id, order_date, payment_method, order_status, order_month, order_dayofweek]"
3,order_items,752,763,6,14,"[order_item_id, order_id, product_id, quantity, unit_price, line_total]"


공식 결과 order_date 재변환 실패: 0 | order_date 결측: 3
공식 결과 category 값: ['도서', '뷰 티', '뷰티', '생활 용품', '생활용품', '스포츠', '식품', '전자 기기', '전자기기', '패션']
공식 스크립트가 삭제한 product_id: [18, 48, 73, 95]
공식 결과 product FK 누락: 26 / 내 규칙: 2
공식 스크립트가 삭제한 주문 상세(중복 제외): {'변환 실패': 4, '0 이하 값': 4, '형식 문제(개)': 3}
공식 결과 age 최댓값: 150.0 | 미래 order_date 포함: 2


In [20]:
# (4) 공식 요약 보고서(ch05_preprocessing_summary.md) 검토: 가이드 STEP 16의 9개 항목 포함 여부
report_text = (REPORT_DIR / 'ch05_preprocessing_summary.md').read_text(encoding='utf-8')
print('[소제목]', [l for l in report_text.splitlines() if l.startswith('#')])
items = {
    '전처리 결과 파일': '전처리 결과 파일', '전처리 전후 데이터 크기': '전후 데이터 크기', '중복 점검': '중복 점검',
    '관계 점검': '관계 점검', '문자열·숫자·날짜 처리 기준': '공백', '결측치 처리 기준': '결측치', '이상값 처리 기준': '0 이하',
    '파생 컬럼': 'line_total', '업무 적용 시 한계': '주의 사항',
}
report_check = pd.DataFrame([{'STEP 16 항목': k, '키워드': v, '포함': v in report_text} for k, v in items.items()])
report_check['내 판단(아래 본문 근거): 몇 건·왜 바꿨는지 설명?'] = ['-', '표로 제시', '표로 제시', '표만 있고 26건 원인 설명 없음',
                                                '규칙만, 건수 없음', '규칙만, 건수 없음', '규칙만, 건수 없음', '규칙만', '일반적 주의만']
display(report_check)
import re
body = report_text.split('## 주요 처리 내용')[1].split('## 주의 사항')[0]
print('\n[## 주요 처리 내용 본문]')
print(body.strip())
print('\n독립 소제목 수(제목 제외):', len([l for l in report_text.splitlines() if l.startswith('## ')]),
      '| 주요 처리 내용에 영향 건수(숫자+건/행/개) 포함:', bool(re.search(r'\d+\s*(건|행|개)', body)),
      '| 관계 단절(삭제 → FK 누락) 설명 포함:', any(w in body for w in ['끊', '누락', '고아', 'orphan']))

raw_hash_after = {name: file_md5(RAW_DIR / f'{name}.csv') for name in FILES}
print('\nraw 원본 파일 변경 없음:', raw_hash_before == raw_hash_after)

[소제목] ['# Chapter 5 데이터 전처리 요약', '## 전처리 결과 파일', '## 전처리 전후 데이터 크기', '## 중복 점검 결과', '## 파일 간 관계 점검 결과', '## 주요 처리 내용', '## 주의 사항']


,STEP 16 항목,키워드,포함,내 판단(아래 본문 근거): 몇 건·왜 바꿨는지 설명?
0,전처리 결과 파일,전처리 결과 파일,True,-
1,전처리 전후 데이터 크기,전후 데이터 크기,True,표로 제시
2,중복 점검,중복 점검,True,표로 제시
3,관계 점검,관계 점검,True,표만 있고 26건 원인 설명 없음
4,문자열·숫자·날짜 처리 기준,공백,True,"규칙만, 건수 없음"
5,결측치 처리 기준,결측치,True,"규칙만, 건수 없음"
6,이상값 처리 기준,0 이하,True,"규칙만, 건수 없음"
7,파생 컬럼,line_total,True,규칙만
8,업무 적용 시 한계,주의 사항,True,일반적 주의만



[## 주요 처리 내용 본문]
- 원본 데이터는 직접 수정하지 않고 복사본을 사용했습니다.
- 문자열 컬럼의 앞뒤 공백을 제거했습니다.
- 고객 나이 결측치는 중앙값으로 대체했습니다.
- 고객 도시 결측치는 Unknown으로 처리했습니다.
- 주문 상태값 표기를 completed, cancelled, refunded 중심으로 통일했습니다.
- 날짜 컬럼을 날짜형으로 변환하고 주문 월/요일 파생 컬럼을 만들었습니다.
- 가격, 수량, 단가를 숫자형으로 변환했습니다.
- 0 이하 가격, 수량, 단가는 정상 분석 대상에서 제외했습니다.
- 주문 상세 금액 line_total 파생 컬럼을 생성했습니다.
- 전처리 후 파일 간 키 관계를 다시 확인했습니다.

독립 소제목 수(제목 제외): 6 | 주요 처리 내용에 영향 건수(숫자+건/행/개) 포함: False | 관계 단절(삭제 → FK 누락) 설명 포함: False

raw 원본 파일 변경 없음: True


- 생성된 clean CSV:
  - 공식 스크립트: `data/processed/customers_clean.csv`(150행), `products_clean.csv`(96행), `orders_clean.csv`(300행), `order_items_clean.csv`(752행)
  - 내 규칙: `data/processed/ch05_notebook/` 4개 파일 + `change_log.csv`
- 생성된 요약 보고서: `reports/ch05_preprocessing_summary.md`
- `python scripts/preprocess_data.py` 재실행 결과:
  - 안내된 명령 그대로 실행하면 **`ModuleNotFoundError: No module named 'src'`**(returncode 1)로 실패합니다. 스크립트를 직접 실행하면 import 경로에 `scripts/` 폴더만 들어가고 프로젝트 루트가 빠지기 때문입니다.
  - 프로젝트 루트를 `PYTHONPATH`에 넣으면 성공합니다(returncode 0). 한 번 더 실행해도 출력 파일 5개의 MD5가 모두 같았습니다.
- 재실행 후 달라진 점:
  - 결과 내용이 그대로였고, 실행 전 `data/processed`에 있던 파일과도 내용이 같습니다.
  - raw 원본 해시도 변하지 않았습니다.
  - **내 규칙**도 `ch05_preprocess.py`로 2번 실행했습니다. 출력 5개 파일이 Notebook 결과와 **바이트 단위로 같았습니다**(MD5 일치).

![재실행 결과](images/step06_reproducible.png)

### 결과 관찰
공식 스크립트 결과와 내 규칙 결과는 다음과 같이 다릅니다.

| 항목 | 공식 스크립트(`src/preprocessing.py`) | 내 규칙 |
| --- | --- | --- |
| products 행 | 96 (가격 문제 상품 18·48·73·95 삭제) | 100 (삭제 없이 플래그) |
| order_items 행 | 752 (형식 문제 `개` 3행 + 변환 실패 4행 + 0 이하 값 4행 = 11행 삭제) | 763 (형식 복구, 실패·이상값은 플래그) |
| product FK 누락 | **26** | 2 |
| category 종류 | 10 (`뷰 티`, `생활 용품`, `전자 기기` 남음) | 7 |
| age 150, 미래 주문일 | 그대로, 표시 없음 | 그대로, 플래그 |

공식 요약 보고서는 STEP 16의 9개 항목에 해당하는 키워드를 모두 포함합니다. 하지만 독립 소제목은 6개(제목 제외)뿐이고, `## 주요 처리 내용` 본문(출력)에는 처리 규칙별 **영향 건수**가 없습니다. 관계 점검표의 **26건이 상품 삭제 때문에 생겼다는 설명**도 없습니다.

### 나의 해석과 판단
- **Notebook에서 한 번 성공하는 것과 스크립트로 재현 가능한 것은 다릅니다.**
  - Notebook은 셀 실행 순서, 커널에 남은 변수, 실행 위치에 따라 결과가 달라질 수 있습니다.
  - 스크립트는 같은 입력 → 같은 출력을 명령 하나로 보장합니다.
  - 그래서 Notebook 규칙 셀과 **같은 코드 조각**으로 `ch05_preprocess.py`를 만들었습니다. 2회 실행 결과가 Notebook 결과와 MD5까지 같은 것을 확인했습니다. 이제 내 규칙도 명령 하나(`python <경로>/ch05_preprocess.py`, 강사 저장소 루트에서 실행)로 재현됩니다.
- 그러나 **재현 가능 ≠ 올바름**입니다.
  - 공식 스크립트는 매번 똑같이 26건의 FK 단절과 표준화되지 않은 카테고리 3종을 만듭니다.
  - 안내된 실행 명령 자체가 import 경로 문제로 실패했습니다. 다른 PC에서 재현하려면 실행 방법(`PYTHONPATH` 설정 또는 `pip install -e .`)까지 문서화해야 합니다.
- 공식 보고서는 "무엇을 바꿨는가"는 있지만 "몇 건을, 왜"가 없습니다. 그래서 내 결과에는 `change_log.csv`(규칙별 영향 건수)를 함께 남겼습니다.

## 7. Chapter 05 최종 판단

### 내가 정한 전처리 원칙 3가지
1. **원본은 읽기만 하고, 바꾼 것은 모두 기록한다.** raw 파일 해시 비교, 변환 실패 원문(`*_raw`) 보존, 규칙별 영향 건수(`change_log.csv`)
2. **형식 문제는 고치고, 의미를 추측해야 하는 값은 고치지 않는다.** 쉼표·`개` 접미사·공백은 복구합니다. `three`, `unknown`, 존재하지 않는 날짜는 결측으로 남깁니다. 금액·날짜·ID는 대체하지 않습니다.
3. **행 삭제는 완전 중복만 한다. 나머지는 플래그로 표시하고 제외 여부는 분석 질문에서 정한다.** 이상값, FK 누락, 날짜 논리 오류 모두 행을 유지했습니다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
**`price > 0` 조건으로 상품 마스터 행을 삭제하는 처리(공식 `preprocess_products`)**
- 상품 4개의 정가가 이상하다는 이유로 상품 행을 지우면, 그 상품이 들어간 정상 거래 24행이 orphan이 됩니다. 공식 결과의 product FK 누락은 26건(24 + 의심 ID 9991·9992의 2건)입니다.
- 이후 상품·카테고리와 merge하면 미매칭으로 빠지거나 카테고리가 NaN이 되어, 카테고리 집계에서 정상 거래가 **조용히** 사라집니다.
- 오류가 나지 않아서 Chapter 04처럼 merge 검증과 총합 검증을 하지 않으면 발견하기 어렵습니다.

### 다음 EDA에서 특히 주의할 데이터 특성
- **분석 기간**: 주문일 2025-07-01 ~ 2026-08-30 (날짜 결측 3건, 미래 날짜 2건 제외). 2026-08은 30일까지만 데이터가 있습니다.
- **대량 수량**: 수량 50(completed)·100(refunded) 행이 있습니다. 전체 completed 금액의 3.4%, 패션 카테고리 안에서는 14.0%입니다. 순위는 그대로지만 패션과 2위의 격차가 절반 이하로 줄어드므로 포함/제외 결과를 함께 봐야 합니다.
- **가입일**: 주문일 < 가입일이 58건(19.3%)이라 `signup_date` 기반 분석은 신뢰할 수 없습니다.
- **고객 속성**: 연령대 분석 시 대체 6명(`age_imputed`)이 모두 40대에 들어가므로 제외 결과를 함께 제시해야 합니다. `city = Unknown` 2명은 별도 그룹입니다.
- **의심 ID**: 참조 대상이 없는 9991·9992 행은 분석에서 제외합니다(테스트·연동 오류로 의심, 확인 전).
- **공식 결과 사용 시**: Chapter 06이 `data/processed`(공식 스크립트 결과)를 쓴다면 category 표기 3종이 남아 있습니다. 집계 전에 다시 표준화하거나 내 규칙 결과(`ch05_notebook/`)를 사용해야 합니다.

### 현재 전처리의 한계
- 이상값과 FK 누락이 실제 오류인지 업무적 의미가 있는지는 원천 시스템이나 담당자 확인 없이는 알 수 없습니다. 그래서 플래그로 보류했습니다.
- 미래 날짜 기준일(작성일 2026-10-15)과 IQR 상한(7)은 업무 기준이 아니라 내가 정한 임시 기준입니다.
- 강사 저장소의 `src/preprocessing.py`는 수정하지 않았습니다. 그래서 공식 경로(`data/processed`)의 결과는 공식 규칙 그대로입니다.

---
## 부록 A. LLM 전처리 제안 검증 (가이드 STEP 17)

### Prompt (원본 행·이름 없이 요약 정보만 제공)
- 사용 LLM: Claude (Anthropic)

```text
고객 데이터 품질 요약입니다. (원본 행이나 이름은 제공하지 않습니다)
- rows: 152 (완전 중복 2행 포함), customer_id 중복 2
- age: 문자열로 저장, 결측 4 + 'unknown' 2 = 6건(4%), 숫자 변환 가능 값은 18~90세와 150세 1건
- 중복 제거 후 150명 기준 중앙값 41, 평균 44.0 (150세 제외 시 43.3)
- 분석 목적: 연령대별 구매 금액 비교

1) age 결측 6건 처리 후보(삭제/중앙값/그룹별 대체/결측 유지)를 비교해 주세요.
2) 각 방법이 연령대별 비교에 주는 영향과 처리 전후 검증 항목을 제안해 주세요.
3) 150세 값은 삭제·대체를 단정하지 말고 확인해야 할 질문을 제시해 주세요.
4) raw를 덮어쓰지 않는 코드로, 변경 건수를 기록하는 방식으로 작성해 주세요.
```

### LLM 응답 (요약 + 코드 원문)
1. **후보 비교**
   - 삭제: 단순하고 가짜 값이 없지만 표본 4%를 잃고, 결측이 특정 집단에 몰려 있으면 편향이 생깁니다.
   - 중앙값 대체: 표본을 유지하고 이상값에 강하지만, 6명이 같은 값을 가져 중앙값이 속한 연령대(40대)가 커지고 분산이 줄어듭니다.
   - 그룹별(예: 성별) 중앙값 대체: 집단 특성을 반영하지만 그룹 기준의 근거가 필요하고, 그룹 표본이 작으면 불안정합니다.
   - 결측 유지('미상' 연령대): 값을 왜곡하지 않지만 연령대 분석에 별도 그룹이 생깁니다.
2. **검증 항목**: 대체 전후 건수, 연령대 분포 비교, 대체 플래그, 원본 보존 여부
3. **150세 확인 질문**: 15세·50세 등의 입력 오타인가? 생년월일 원천 값은 무엇인가? 나이 계산 기준일은 언제인가? → 확인 전까지 분석 제외 플래그를 권장합니다.
4. **코드**: 아래 셀의 `LLM_CODE`(원문 그대로)

**검증 순서**
1. 코드 원문을 Raw에 그대로 실행합니다.
2. 가이드 STEP 17의 사람 확인 7개 항목으로 판정합니다.
3. 응답의 4개 후보를 실제 데이터에 적용해 비교합니다.

In [21]:
# LLM 제안 코드(원문)를 Raw에 그대로 실행 → 결과 확인
LLM_CODE = r"""
customers_raw = pd.read_csv(RAW_DIR / "customers.csv")
customers = customers_raw.copy()                       # 원본은 그대로 두고 복사본에서 처리

customers["age"] = pd.to_numeric(customers["age"], errors="coerce")
print("age 숫자 변환 후 결측:", customers["age"].isna().sum())

customers.loc[customers["age"] > 120, "age"] = np.nan   # 비현실적인 나이는 결측으로 처리
median_age = customers["age"].median()
customers["age_imputed"] = customers["age"].isna()
customers["age"] = customers["age"].fillna(median_age)
print("대체 건수:", customers["age_imputed"].sum(), "| 사용한 중앙값:", median_age)
"""
llm_ns = {'pd': pd, 'np': np, 'RAW_DIR': RAW_DIR}
exec(LLM_CODE, llm_ns)
llm_c = llm_ns['customers']
print('\n[검증] LLM 결과 행 수:', len(llm_c), '| 중복 행:', int(llm_c.duplicated(subset=['customer_id']).sum()),
      '| customer 33의 age:', llm_c.loc[llm_c['customer_id'] == 33, 'age'].tolist(),
      '| raw 원본 변경:', file_md5(RAW_DIR / 'customers.csv') != raw_hash_before['customers'])

age 숫자 변환 후 결측: 6
대체 건수: 7 | 사용한 중앙값: 41.0

[검증] LLM 결과 행 수: 152 | 중복 행: 2 | customer 33의 age: [41.0] | raw 원본 변경: False


In [22]:
# 가이드 STEP 17 사람 확인 7개 항목으로 LLM 제안 판정
llm_review = pd.DataFrame([
    ['raw를 덮어쓰는가?', 'O(안전)', 'copy()에서 처리, 원본 파일 해시 변경 없음'],
    ['변환 실패를 확인하는가?', '△', '변환 후 결측 6건만 출력 — 원래 결측 4건과 변환 실패 2건(unknown)을 구분하지 않고 원문 보존도 없음'],
    ['결측 처리 근거가 있는가?', 'O', '4개 후보 비교 후 중앙값 선택 이유(이상값에 강함)를 제시'],
    ['키 중복을 자동 삭제하지 않는가?', '△', f'삭제는 안 하지만 중복 처리 없이 {len(llm_c)}행 기준으로 중앙값 계산(내 코드도 대체를 중복 제거 전에 함 — 150행 기준 중앙값도 41.0이라 결과 영향 없음을 2번에서 확인)'],
    ['이상값을 자동 삭제·수정하지 않는가?', 'X', '150세를 자동으로 NaN → 중앙값 41로 바꿈. 답변 3번의 "확인 전까지 플래그" 권고와 코드가 모순'],
    ['관계를 재검증하는가?', '해당 없음', 'customers 행을 삭제하지 않으므로 FK 영향 없음(본 Notebook 4번에서 별도 확인)'],
    ['변경 내역을 기록하는가?', '△', 'age_imputed와 대체 건수는 남지만 150세 변경은 기록되지 않음'],
], columns=['STEP 17 확인 항목', '판정', '근거'])
display(llm_review)

# LLM이 제안한 4개 후보를 실제 데이터에 적용해 비교 (중복 제거 후 150명 기준)
base = raw['customers'].drop_duplicates().copy()
base['age'] = pd.to_numeric(base['age'], errors='coerce')
base = base[~(base['age'] > AGE_VALID_MAX)]                  # 150세 1명은 비교에서 제외(이상값 후보는 별도 플래그 처리)
na = base['age'].isna()
cut = lambda s: pd.cut(s, bins=[0, 30, 40, 50, 60, 200], right=False, labels=['~20대', '30대', '40대', '50대', '60대+'])   # 39.5세 → 30대
g_med = base.groupby('gender')['age'].median()
opt = {
    '① 삭제': cut(base.loc[~na, 'age']).value_counts(),
    '② 중앙값 대체': cut(base['age'].fillna(base['age'].median())).value_counts(),
    '③ 성별 중앙값 대체': cut(base['age'].fillna(base['gender'].map(g_med))).value_counts(),
    '④ 결측 유지(미상)': cut(base['age']).value_counts(),
}
opt_df = pd.DataFrame(opt).sort_index()
opt_df.loc['미상/제외'] = [int(na.sum()), 0, 0, int(na.sum())]
display(opt_df)
all_age = pd.to_numeric(raw['customers'].drop_duplicates()['age'], errors='coerce')
print(f"150세 포함/제외 — 평균 {all_age.mean():.2f} / {all_age[all_age <= AGE_VALID_MAX].mean():.2f}, 중앙값 {all_age.median()} / {all_age[all_age <= AGE_VALID_MAX].median()}")
print('성별 중앙값:', g_med.to_dict(), '| 결측 6명 성별:', base.loc[na, 'gender'].value_counts().to_dict(),
      '| 결측 고객의 주문 수:', int(clean['orders']['customer_id'].isin(base.loc[na, 'customer_id']).sum()))

,STEP 17 확인 항목,판정,근거
0,raw를 덮어쓰는가?,O(안전),"copy()에서 처리, 원본 파일 해시 변경 없음"
1,변환 실패를 확인하는가?,△,변환 후 결측 6건만 출력 — 원래 결측 4건과 변환 실패 2건(unknown)을 구분하지 않고 원문 보존도 없음
2,결측 처리 근거가 있는가?,O,4개 후보 비교 후 중앙값 선택 이유(이상값에 강함)를 제시
3,키 중복을 자동 삭제하지 않는가?,△,삭제는 안 하지만 중복 처리 없이 152행 기준으로 중앙값 계산(내 코드도 대체를 중복 제거 전에 함 — 150행 기준 중앙값도 41.0이라 결과 영향 없음을 2번에서 확인)
4,이상값을 자동 삭제·수정하지 않는가?,X,"150세를 자동으로 NaN → 중앙값 41로 바꿈. 답변 3번의 ""확인 전까지 플래그"" 권고와 코드가 모순"
5,관계를 재검증하는가?,해당 없음,customers 행을 삭제하지 않으므로 FK 영향 없음(본 Notebook 4번에서 별도 확인)
6,변경 내역을 기록하는가?,△,age_imputed와 대체 건수는 남지만 150세 변경은 기록되지 않음


,① 삭제,② 중앙값 대체,③ 성별 중앙값 대체,④ 결측 유지(미상)
age,,,,
~20대,35,35,35,35
30대,31,31,34,31
40대,24,30,27,24
50대,23,23,23,23
60대+,30,30,30,30
미상/제외,6,0,0,6


150세 포함/제외 — 평균 44.03 / 43.29, 중앙값 41.0 / 41.0
성별 중앙값: {'F': 39.5, 'M': 44.0} | 결측 6명 성별: {'M': 3, 'F': 3} | 결측 고객의 주문 수: 9


![LLM 제안 검증](images/step07_llm_review.png)

### 결과 관찰
- LLM 코드를 실행하자 "age 숫자 변환 후 결측 6"을 출력한 뒤 **대체 건수는 7**이 되었습니다. 150세를 결측으로 바꾼 1건이 더해졌기 때문입니다. customer 33의 나이는 150 → 41.0으로 바뀌었습니다.
- LLM 코드는 중복 2행이 남은 152행에서 실행되었습니다. raw 원본 파일은 바뀌지 않았습니다.
- 4개 후보를 실제 데이터에 적용한 결과:
  - 비교 기준은 150세 1명을 뺀 149명입니다. 그래서 150세를 플래그만 하고 포함한 5번 민감도 표보다 60대+가 1명 적습니다(30 vs 31).
  - 연령대 구간은 `[30, 40)`처럼 왼쪽을 포함해 나눴습니다. 그래서 39.5세는 30대로 분류됩니다.
  - ① 삭제와 ④ 결측 유지: 40대 24명, 미상·제외 6명
  - ② 중앙값 대체: 6명이 모두 40대로 들어가 40대 30명
  - ③ 성별 중앙값 대체(F 39.5, M 44.0): 여성 3명은 30대, 남성 3명은 40대로 나뉘어 30대 34명, 40대 27명
  - 결측 6명은 남녀 3명씩이고, 이들의 주문은 9건입니다.

### 나의 해석과 판단
- **채택한 것**: 4개 후보 비교와 "중앙값 + 대체 플래그" 방식. 이상값에 흔들리지 않는다는 근거가 실제 데이터로 확인됐습니다. 평균은 150세 때문에 0.74세 차이가 났고, 중앙값은 0이었습니다.
- **채택하지 않은 것**: 150세를 자동으로 결측 → 41세로 바꾸는 코드. LLM 스스로 답변 3번에서 "확인 전까지 플래그"를 권했는데, 코드는 값을 바꿔 버려 **설명과 코드가 서로 모순**됩니다. 실행 결과(대체 7건, customer 33 = 41.0)로 이를 확인했습니다. 나는 150세를 플래그로만 표시했습니다(3번).
- **보완한 것**: 변환 실패 원문 보존(`age_raw`), 150세 변경 금지(플래그만). 중복 행 문제는 내 코드도 같은 순서(대체 후 중복 제거)라 LLM만 감점하지 않았고, 150행 기준 중앙값도 41.0이라 결과 영향이 없음을 2번에서 확인했습니다.
- ③ 성별 중앙값은 40대 쏠림을 30대·40대로 나눠 줍니다(+3, +3). 하지만 "성별이 나이를 설명한다"는 근거가 없고, 나이와 성별의 관계를 확인한 근거도 없습니다. 그래서 **보류**하고, 단순한 ②에 `age_imputed` 플래그를 붙여 쏠림을 분석 단계에서 분리하는 쪽을 택했습니다.
- 결론: LLM 답변의 설명은 대체로 타당했지만, **코드가 설명대로 동작하는지는 실행해 봐야 알 수 있었습니다.** 판정은 "부분 채택"입니다.

---
## 부록 B. 공식 Notebook 실습 과제(24번)와 LLM 검토 예시 코드(23번)

In [23]:
c, o, oi = clean['customers'], clean['orders'], clean['order_items']
# 과제 1. 데이터셋별 결측 비율 (원 컬럼 기준, raw vs clean)
miss_tbl = pd.concat([pd.DataFrame({'dataset': n, 'column': raw[n].columns,
                                    'raw_missing(%)': (raw[n].isna().mean() * 100).round(2).values,
                                    'clean_missing(%)': (clean[n][raw[n].columns].isna().mean() * 100).round(2).values}) for n in FILES])
display(miss_tbl[(miss_tbl['raw_missing(%)'] > 0) | (miss_tbl['clean_missing(%)'] > 0)])
# 과제 2. 18세 미만 또는 100세 초과
print('18세 미만 또는 100세 초과:', c.loc[(c['age'] < 18) | (c['age'] > 100), ['customer_id', 'age', 'age_imputed']].to_dict('records'))
# 과제 3. 월별 주문 건수 (날짜 결측·미래 날짜 제외)
monthly_orders = o[~o['flag_future_date'] & o['order_month'].notna()].groupby('order_month')['order_id'].nunique()
print('월별 주문 건수:', monthly_orders.to_dict())
print('합계:', monthly_orders.sum(), '/ 전체', len(o), '(날짜 결측', int(o['order_month'].isna().sum()), ', 미래', int(o['flag_future_date'].sum()), ')')
# 과제 4. line_total 상위 10
display(oi.nlargest(10, 'line_total')[['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total', 'flag_qty_large']])

,dataset,column,raw_missing(%),clean_missing(%)
3,customers,age,2.63,0.00
4,customers,city,1.32,0.00
5,customers,signup_date,0.00,1.33
3,products,price,0.00,2.00
2,orders,order_date,0.00,1.00
3,order_items,quantity,0.00,0.26
4,order_items,unit_price,0.00,0.26


18세 미만 또는 100세 초과: [{'customer_id': 33, 'age': 150.0, 'age_imputed': False}]
월별 주문 건수: {'2025-07': 19, '2025-08': 26, '2025-09': 12, '2025-10': 22, '2025-11': 23, '2025-12': 23, '2026-01': 26, '2026-02': 19, '2026-03': 22, '2026-04': 26, '2026-05': 20, '2026-06': 24, '2026-07': 12, '2026-08': 21}
합계: 295 / 전체 300 (날짜 결측 3 , 미래 2 )


,order_item_id,order_id,product_id,quantity,unit_price,line_total,flag_qty_large
750,751,295,31,100.0,155000.0,15500000.0,True
737,738,290,96,50.0,96000.0,4800000.0,True
319,320,129,23,5.0,197000.0,985000.0,False
9,10,5,65,5.0,193000.0,965000.0,False
20,21,9,36,5.0,193000.0,965000.0,False
24,25,11,65,5.0,193000.0,965000.0,False
116,117,51,65,5.0,193000.0,965000.0,False
293,294,119,36,5.0,193000.0,965000.0,False
290,291,117,47,5.0,188000.0,940000.0,False
10,11,6,97,5.0,187000.0,935000.0,False


In [24]:
# 공식 Notebook 23번: LLM 검토 요청 예시 코드를 Raw에 그대로 실행
res = []
for label, stmt in [('age 평균 대체', "c['age'] = c['age'].fillna(c['age'].mean())"),
                    ('날짜 변환(coerce 없음)', "o['order_date'] = pd.to_datetime(o['order_date'])"),
                    ('line_total(숫자 변환 없음)', "oi['line_total'] = oi['quantity'] * oi['unit_price']"),
                    ('중복 제거', "c = c.drop_duplicates()")]:
    ns = {'pd': pd, 'c': customers.copy(), 'o': orders.copy(), 'oi': order_items.copy()}
    try:
        exec(stmt, ns)
        out = '실행됨' + (f" → {len(customers)}→{len(ns['c'])}행 (완전 중복만 제거, 제거 내역·key 중복 점검 없음)" if label == '중복 제거' else '')
    except Exception as e:
        out = f'{type(e).__name__}: {str(e).splitlines()[0][:90]}'
    res.append({'코드': label, '결과': out})
display(pd.DataFrame(res))

,코드,결과
0,age 평균 대체,TypeError: Cannot perform reduction 'mean' with string dtype
1,날짜 변환(coerce 없음),"ValueError: time data ""not-a-date"" doesn't match format ""%Y-%m-%d"". You might want to try:"
2,line_total(숫자 변환 없음),TypeError: can't multiply sequence by non-int of type 'str'
3,중복 제거,"실행됨 → 152→150행 (완전 중복만 제거, 제거 내역·key 중복 점검 없음)"


- 과제 1: raw 결측 비율은 `age` 2.63%, `city` 1.32%입니다. clean에서는 둘 다 0%가 되고, 대신 변환 실패에서 나온 결측(날짜·가격·수량·단가 0.26 ~ 2.00%)이 드러납니다.
- 과제 2: 18세 미만은 없습니다. 100세 초과는 customer 33(150세) 1명입니다(대체값 아님, 플래그 대상).
- 과제 3: 날짜 결측 3건과 미래 날짜 2건을 뺀 295건의 월별 주문 건수는 2025-07 ~ 2026-08, 월 12 ~ 26건입니다.
- 과제 4: `line_total` 1·2위가 수량 이상값 후보(100개 15,500,000원, 50개 4,800,000원)입니다. 3위(985,000원)와 차이가 커서 합계를 왜곡할 수 있습니다.
- 과제 5: 내 규칙에서는 4개 테이블 모두 2행씩(완전 중복) 줄었습니다. 공식 스크립트는 products −6, order_items −13으로 더 많이 줄입니다.
- 과제 6: 부록 A의 Prompt

**LLM 검토 예시 코드(23번)를 Raw에 그대로 실행한 결과**
- `fillna(mean())`: age가 문자열이라 **TypeError**가 납니다.
- `pd.to_datetime(order_date)`: `not-a-date`에서 **ValueError**가 납니다. `errors='coerce'`도, 실패 건수 확인도 없습니다.
- `quantity * unit_price`: 둘 다 문자열이라 **TypeError**가 납니다.
- `drop_duplicates()`: 실행은 되지만 제거 내역 기록도, key 중복 점검도 없습니다.
- 판단: 4줄 중 3줄은 실제 데이터에서 실행조차 되지 않았습니다. 깨끗한 데이터에서 실행되는 코드라도 실제 데이터에서 다시 검증해야 합니다.

---
## 최종 제출 체크
- [x] raw 원본을 덮어쓰지 않았습니다. (처음·마지막 MD5 동일)
- [x] 처리 전/후 비교를 남겼습니다. (5번 표, 민감도, `change_log.csv`)
- [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다. (공식 스크립트 2회 해시 동일, 내 규칙 스크립트 2회 = Notebook 결과)
- [x] 개인정보/Secret이 없습니다. (`name` 미출력·미저장, API Key 없음)
- [x] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.